In [ ]:
!pip -q install openai pandas numpy scipy scikit-learn tqdm pillow matplotlib


In [ ]:
from google.colab import drive, userdata
from IPython.display import Markdown, display
from pathlib import Path
from datetime import datetime, timezone
import base64
import hashlib
import json
import os
import random
import re
import time
from typing import Optional

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import rankdata, spearmanr
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_absolute_error
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from tqdm.auto import tqdm
from openai import OpenAI

RANDOM_SEED = 20260731
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

                                                                                
RUN_NEW_API_CALLS = True
RUN_LIMIT = None
ALLOW_INCOMPLETE_CACHE_DIAGNOSTICS = True

DIRECT_MODEL = 'gpt-5.4'
DIRECT_REASONING_EFFORT = 'none'
DIRECT_IMAGE_DETAIL = 'original'
STAGE_1_MODEL = 'gpt-5.4'
STAGE_2_MODEL = 'gpt-5.4'
STAGE_2_REASONING_EFFORT = 'none'
STAGE_2_IMAGE_DETAIL = 'original'

RIDGE_ALPHAS = [0.1, 1.0, 10.0, 100.0]
OUTER_FOLDS = 5
INNER_FOLDS = 4
RESIDUAL_LIMIT = 0.20                                                  
NEAR_ZERO_THRESHOLD = 0.10
BOOTSTRAP_REPLICATES = 2000
REQUEST_SLEEP_SECONDS = 0.2
MAX_RETRIES = 3

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 160)

drive.mount('/content/drive')
IMAGEEVAL_ROOT = Path('/content/drive/MyDrive/Dr. Lulwah - Ahmed/ImageEVAl')
PROJECT_DIR = IMAGEEVAL_ROOT / 'ImageEval2026_Task2_CRAI_Bench'
DATA_DIR_CANDIDATES = [PROJECT_DIR / 'data', IMAGEEVAL_ROOT / 'train_dev']
DATA_DIR = next(
    (path for path in DATA_DIR_CANDIDATES
     if (path / 'train' / 'captions.tsv').exists()),
    DATA_DIR_CANDIDATES[0],
)

EXPERIMENT_ROOT = PROJECT_DIR / 'cea_structured_compact_v1'
NEW_CACHE_DIR = EXPERIMENT_ROOT / 'cache'
OUTPUT_DIR = EXPERIMENT_ROOT / 'outputs'
PERSISTENT_ROOT = PROJECT_DIR / 'cea_persistent_anchor_diagnostic'
PERSISTENT_CACHE_DIR = PERSISTENT_ROOT / 'cache'
HISTORICAL_DIRECT_CACHE_DIR = PROJECT_DIR / 'cea_architecture_comparison' / 'cache'
for path in [NEW_CACHE_DIR, OUTPUT_DIR]:
    path.mkdir(parents=True, exist_ok=True)

                                                                                      
CACHE_DIR = PERSISTENT_CACHE_DIR
client = None

def get_openai_client():
    global client
    if client is None:
        if not RUN_NEW_API_CALLS:
            raise RuntimeError('API calls are disabled. Set RUN_NEW_API_CALLS=True explicitly.')
        key = userdata.get('openai')
        if not key:
            raise RuntimeError('Colab secret "openai" is missing.')
        os.environ['OPENAI_API_KEY'] = key
        client = OpenAI()
    return client

print('Data:', DATA_DIR)
print('New experiment:', EXPERIMENT_ROOT)
print('API calls enabled:', RUN_NEW_API_CALLS)


In [ ]:
DIM_COLS = ['CRAI_CEA']

def parse_image_base_id(instance_id: str) -> str:
    return re.sub(r'_v\d+$', '', str(instance_id))

def parse_caption_version(instance_id: str) -> int:
    match = re.search(r'_v(\d+)$', str(instance_id))
    return int(match.group(1)) if match else -1

def find_existing_image(folder: Path, stem: str) -> Path:
    for extension in ['.png', '.jpg', '.jpeg', '.webp']:
        candidate = folder / f'{stem}{extension}'
        if candidate.exists():
            return candidate
    raise FileNotFoundError(f'No image found for {stem} in {folder}')

def load_split(split: str, require_gold: bool = True) -> pd.DataFrame:
    split_dir = DATA_DIR / split
    captions = pd.read_csv(split_dir / 'captions.tsv', sep='\t')
    frame = captions.copy()
    if require_gold:
        gold = pd.read_csv(split_dir / 'gold_human.tsv', sep='\t')
        frame = frame.merge(gold, on='id', how='left', validate='one_to_one')
    frame['id'] = frame['id'].astype(str)
    frame['split'] = split
    frame['base_id'] = frame['id'].map(parse_image_base_id)
    frame['caption_version'] = frame['id'].map(parse_caption_version)
    frame['caption_version_key'] = frame['caption_version'].map(lambda x: f'v{x}')
    if 'category' not in frame.columns:
        frame['category'] = 'unknown'
    frame['category'] = frame['category'].fillna('unknown').astype(str)
    frame['ref_image_path'] = frame['base_id'].map(
        lambda value: str(find_existing_image(split_dir / 'imgs' / 'ref', value))
    )
    frame['generated_image_path'] = frame['id'].map(
        lambda value: str(find_existing_image(split_dir / 'imgs' / 'generated', value))
    )
    return frame

def assert_group_structure(frame: pd.DataFrame, split: str):
    assert frame['id'].is_unique, f'{split}: duplicate instance IDs'
    assert frame['base_id'].notna().all(), f'{split}: missing group IDs'
    assert frame.groupby('id')['base_id'].nunique().eq(1).all()
    versions = frame.groupby('base_id')['caption_version'].apply(lambda x: set(map(int, x)))
    bad = versions[versions != {1, 2, 3, 4, 5}]
    assert bad.empty, f'{split}: incomplete/mixed caption versions: {bad.to_dict()}'
    assert frame.groupby('base_id')['ref_image_path'].nunique().eq(1).all()

train_df = load_split('train', require_gold=True)
dev_df = load_split('dev', require_gold=True)
assert_group_structure(train_df, 'train')
assert_group_structure(dev_df, 'dev')
assert not set(train_df['base_id']) & set(dev_df['base_id'])

                                                                                
                                                                                
train_caption_columns = set(pd.read_csv(DATA_DIR / 'train' / 'captions.tsv', sep='\t', nrows=1).columns)
dev_caption_columns = set(pd.read_csv(DATA_DIR / 'dev' / 'captions.tsv', sep='\t', nrows=1).columns)
USE_CATEGORY_FEATURE = 'category' in (train_caption_columns & dev_caption_columns)
print('train:', len(train_df), 'rows |', train_df['base_id'].nunique(), 'groups')
print('dev:', len(dev_df), 'rows |', dev_df['base_id'].nunique(), 'groups')
print('Category used as model feature:', USE_CATEGORY_FEATURE)
display(train_df[['id', 'base_id', 'caption_version_key', 'category', 'CRAI_CEA']].head())


In [ ]:
def prompt_hash(text: str, length: int = 10) -> str:
    return hashlib.sha256(text.strip().encode('utf-8')).hexdigest()[:length]

def image_to_data_url(path: str) -> str:
    path = Path(path)
    suffix = path.suffix.lower()
    media_type = {
        '.png': 'image/png', '.jpg': 'image/jpeg', '.jpeg': 'image/jpeg',
        '.webp': 'image/webp',
    }.get(suffix)
    if media_type is None:
        raise ValueError(f'Unsupported image type: {path}')
    payload = base64.b64encode(path.read_bytes()).decode('ascii')
    return f'data:{media_type};base64,{payload}'

def extract_caption(row: pd.Series) -> str:
    for column in ['caption', 'text', 'prompt']:
        if column in row and pd.notna(row[column]):
            return str(row[column])
    raise KeyError('No caption column found')

def parse_json_object(text: str) -> dict:
    text = str(text).strip()
    if text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*', '', text, flags=re.I)
        text = re.sub(r'\s*```$', '', text)
    start, end = text.find('{'), text.rfind('}')
    if start < 0 or end < start:
        raise ValueError('Response did not contain a JSON object')
    value = json.loads(text[start:end + 1])
    if not isinstance(value, dict):
        raise ValueError('Expected a JSON object')
    return value

def load_jsonl_records(path: Path) -> list[dict]:
    if not path.exists():
        return []
    output = []
    with path.open(encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, start=1):
            if line.strip():
                try:
                    output.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(f'Malformed JSONL at {path}:{line_number}') from exc
    return output

def index_by_key(records, key: str, label: str) -> dict:
    output = {}
    for record in records:
        value = str(record[key])
        if value in output:
            raise ValueError(f'Duplicate {key}={value!r} in {label}')
        output[value] = record
    return output

def append_jsonl(path: Path, record: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + '\n')
        handle.flush()

def cache_coverage(frame: pd.DataFrame, records, key='instance_id') -> dict:
    requested = frame['id'].astype(str).tolist()
    available = {str(record[key]) for record in records}
    missing = [value for value in requested if value not in available]
    return {'expected': len(requested), 'available': len(requested) - len(missing), 'missing': missing}

HISTORICAL_CACHE_MANIFEST = pd.DataFrame([
    {'system': 'persistent_inventory', 'compatible': True,
     'configuration': 'gpt-5.5 / prompt 36a9d72a2f'},
    {'system': 'persistent_requirements', 'compatible': True,
     'configuration': 'gpt-5.5 / inventory 36a9d72a2f / prompt 924e4d307b'},
    {'system': 'persistent_stage2', 'compatible': True,
     'configuration': 'gpt-5.5 / reasoning none / prompt 5aed554e3e'},
    {'system': 'old_direct_hybrid_d', 'compatible': False,
     'configuration': 'gpt-5.5 / prompt 68f12070e9 / demos img_022_v3,img_035_v4'},
])
display(HISTORICAL_CACHE_MANIFEST)


In [ ]:
DIRECT_PROMPT_VERSION = 'structured-direct-cea-v1'
DIRECT_SCHEMA_VERSION = 'structured-direct-cea-schema-v1'
DIRECT_DEMONSTRATION_IDS = []
DIRECT_DEMONSTRATION_RULE = 'zero-shot; no gold-labelled demonstrations'

STRUCTURED_DIRECT_CEA_PROMPT = r"""
ROLE

You are a strict multimodal judge of Cultural Element Accuracy (CEA) for
CRAI-Bench. Return valid JSON only.

CENTRAL RUBRIC

Infer the intended cultural target jointly from the reference image and the v1
caption. The current caption controls the requested scene, but it does not erase
cultural identity established by the reference. Evaluate only culturally
discriminative content for CEA. Do not reward ordinary caption adherence unless it
provides evidence of the intended cultural identity. A generic substitute that has
a similar shape or function must not receive full credit for a culturally specific
landmark, object, practice, clothing style, or setting.

SCORING RULES

- Create only two to four culturally discriminative anchors.
- Do not create anchors for generic image quality, composition, realism, lighting,
  pose, water, sky, or ordinary objects unless they carry cultural identity here.
- Use the reference image and v1 caption jointly. The current caption sets scene
  scope but cannot erase an established cultural identity.
- Missing or incorrect identity must limit that anchor's score.
- Visible generic geometry or function must not compensate for incorrect identity.
- Mark generic_substitution when the candidate contains a generic lookalike/function
  but not the culturally specific target.
- Mark wrong_culture_or_landmark when the candidate depicts a conflicting cultural
  identity or a different recognizable landmark.
- raw_cea is a holistic visual judgment, not a mechanical average of anchor fields.
- Do not score CC, CS, CI, HP, or general caption compliance.
- Keep brief_reason to one short sentence.

QATAR-AWARE INTERPRETATION

When supported by the reference/v1, discriminate Qatari identities such as Katara
Towers, Al Fanar, the Museum of Islamic Art, the National Museum of Qatar, Souq
Waqif, Doha landmark forms, thobe, ghutra, agal, abaya, shayla, battoulah,
falconry, dhow and pearl-diving heritage, majlis settings, sadu weaving, and local
market handicrafts. These are examples, not a checklist. Never invent or require
one merely because the task concerns Qatar.

OUTPUT JSON

{
  "raw_cea": 0.0,
  "confidence": 0.0,
  "caption_scope": "generic | regional | exact_named",
  "anchors": [
    {
      "name": "short culturally discriminative anchor",
      "importance": 0.0,
      "presence": 0.0,
      "identity_match": 0.0,
      "cultural_correctness": 0.0,
      "generic_substitution": false,
      "wrong_culture_or_landmark": false
    }
  ],
  "brief_reason": "one short sentence"
}

All numeric fields are continuous numbers from 0 to 1. Return no markdown and no
text outside the JSON object.
"""

DIRECT_REQUIRED_TOP_LEVEL = {
    'raw_cea', 'confidence', 'caption_scope', 'anchors', 'brief_reason'
}
DIRECT_REQUIRED_ANCHOR = {
    'name', 'importance', 'presence', 'identity_match',
    'cultural_correctness', 'generic_substitution',
    'wrong_culture_or_landmark',
}

def direct_cache_tag() -> str:
    demo_signature = prompt_hash(json.dumps({
        'ids': DIRECT_DEMONSTRATION_IDS,
        'rule': DIRECT_DEMONSTRATION_RULE,
    }, sort_keys=True))
    return (
        f'{DIRECT_PROMPT_VERSION}_{DIRECT_MODEL}_reasoning-{DIRECT_REASONING_EFFORT}_'
        f'detail-{DIRECT_IMAGE_DETAIL}_prompt-{prompt_hash(STRUCTURED_DIRECT_CEA_PROMPT)}_'
        f'schema-{prompt_hash(DIRECT_SCHEMA_VERSION)}_demos-{demo_signature}'
    )

def direct_cache_path(split: str) -> Path:
    return NEW_CACHE_DIR / f'structured_direct_{split}_{direct_cache_tag()}.jsonl'

def direct_attempt_cache_path(split: str) -> Path:
    return NEW_CACHE_DIR / f'structured_direct_attempts_{split}_{direct_cache_tag()}.jsonl'

print('New direct configuration:', direct_cache_tag())
print('Demonstrations:', DIRECT_DEMONSTRATION_IDS, '|', DIRECT_DEMONSTRATION_RULE)


In [ ]:
def finite_unit_interval(value, name: str) -> float:
    number = float(value)
    if not np.isfinite(number) or not 0.0 <= number <= 1.0:
        raise ValueError(f'{name} must be finite and in [0, 1]; got {value!r}')
    return number

def validate_structured_direct_response(value: dict, instance_id: str) -> dict:
    missing = DIRECT_REQUIRED_TOP_LEVEL - set(value)
    if missing:
        raise ValueError(f'Missing top-level fields: {sorted(missing)}')
    scope = str(value['caption_scope'])
    if scope not in {'generic', 'regional', 'exact_named'}:
        raise ValueError(f'Invalid caption_scope: {scope!r}')
    anchors = value['anchors']
    if not isinstance(anchors, list) or not 2 <= len(anchors) <= 4:
        raise ValueError('anchors must contain two to four items')
    canonical_anchors = []
    for position, anchor in enumerate(anchors, start=1):
        missing = DIRECT_REQUIRED_ANCHOR - set(anchor)
        if missing:
            raise ValueError(f'Anchor {position} missing fields: {sorted(missing)}')
        name = str(anchor['name']).strip()
        if not name:
            raise ValueError(f'Anchor {position} has an empty name')
        generic = anchor['generic_substitution']
        wrong = anchor['wrong_culture_or_landmark']
        if not isinstance(generic, bool) or not isinstance(wrong, bool):
            raise ValueError(f'Anchor {position} boolean fields must be JSON booleans')
        canonical_anchors.append({
            'name': name,
            'importance': finite_unit_interval(anchor['importance'], f'anchor_{position}.importance'),
            'presence': finite_unit_interval(anchor['presence'], f'anchor_{position}.presence'),
            'identity_match': finite_unit_interval(anchor['identity_match'], f'anchor_{position}.identity_match'),
            'cultural_correctness': finite_unit_interval(
                anchor['cultural_correctness'], f'anchor_{position}.cultural_correctness'
            ),
            'generic_substitution': generic,
            'wrong_culture_or_landmark': wrong,
        })
    if sum(anchor['importance'] for anchor in canonical_anchors) <= 0:
        raise ValueError('At least one anchor must have positive importance')
    reason = str(value['brief_reason']).strip()
    if not reason:
        raise ValueError('brief_reason is empty')
    return {
        'raw_cea': finite_unit_interval(value['raw_cea'], 'raw_cea'),
        'confidence': finite_unit_interval(value['confidence'], 'confidence'),
        'caption_scope': scope,
        'anchors': canonical_anchors,
        'brief_reason': reason,
    }

def v1_caption_map(frame: pd.DataFrame) -> dict:
    v1 = frame.loc[frame['caption_version'].eq(1)]
    assert not v1['base_id'].duplicated().any()
    return {str(row['base_id']): extract_caption(row) for _, row in v1.iterrows()}

def structured_direct_user_content(row: pd.Series, v1_caption: str) -> list[dict]:
    text = f"""Instance ID: {row['id']}
V1 culturally explicit caption:
{v1_caption}

Current caption (v{row['caption_version']}):
{extract_caption(row)}

The first image is the reference. The second image is the generated candidate.
Return the required JSON only."""
    return [
        {'type': 'input_text', 'text': text},
        {'type': 'input_text', 'text': 'REFERENCE IMAGE:'},
        {'type': 'input_image', 'image_url': image_to_data_url(row['ref_image_path']),
         'detail': DIRECT_IMAGE_DETAIL},
        {'type': 'input_text', 'text': 'GENERATED IMAGE:'},
        {'type': 'input_image', 'image_url': image_to_data_url(row['generated_image_path']),
         'detail': DIRECT_IMAGE_DETAIL},
    ]

def call_structured_direct(row: pd.Series, v1_caption: str, split: str) -> dict:
    for attempt in range(1, MAX_RETRIES + 1):
        timestamp = datetime.now(timezone.utc).isoformat()
        try:
            response = get_openai_client().responses.create(
                model=DIRECT_MODEL,
                reasoning={'effort': DIRECT_REASONING_EFFORT},
                input=[
                    {'role': 'developer', 'content': STRUCTURED_DIRECT_CEA_PROMPT.strip()},
                    {'role': 'user', 'content': structured_direct_user_content(row, v1_caption)},
                ],
            )
            parsed = parse_json_object(response.output_text)
            canonical = validate_structured_direct_response(parsed, str(row['id']))
            record = {
                'instance_id': str(row['id']),
                'prompt_version': DIRECT_PROMPT_VERSION,
                'prompt_hash': prompt_hash(STRUCTURED_DIRECT_CEA_PROMPT),
                'schema_version': DIRECT_SCHEMA_VERSION,
                'model': DIRECT_MODEL,
                'reasoning_effort': DIRECT_REASONING_EFFORT,
                'image_detail': DIRECT_IMAGE_DETAIL,
                'demonstration_ids': DIRECT_DEMONSTRATION_IDS,
                'demonstration_rule': DIRECT_DEMONSTRATION_RULE,
                'parsing_status': 'valid',
                'created_utc': timestamp,
                'response': canonical,
            }
            append_jsonl(direct_attempt_cache_path(split), {
                **{k: record[k] for k in record if k != 'response'},
                'attempt': attempt, 'raw_response': response.output_text,
            })
            return record
        except Exception as exc:
            append_jsonl(direct_attempt_cache_path(split), {
                'instance_id': str(row['id']), 'attempt': attempt,
                'prompt_version': DIRECT_PROMPT_VERSION,
                'prompt_hash': prompt_hash(STRUCTURED_DIRECT_CEA_PROMPT),
                'schema_version': DIRECT_SCHEMA_VERSION, 'model': DIRECT_MODEL,
                'parsing_status': 'invalid', 'error': repr(exc),
                'created_utc': timestamp,
            })
            if attempt == MAX_RETRIES:
                raise
            time.sleep(2 ** attempt)

def validate_direct_cache_record(record: dict) -> dict:
    expected = {
        'prompt_version': DIRECT_PROMPT_VERSION,
        'prompt_hash': prompt_hash(STRUCTURED_DIRECT_CEA_PROMPT),
        'schema_version': DIRECT_SCHEMA_VERSION,
        'model': DIRECT_MODEL,
        'reasoning_effort': DIRECT_REASONING_EFFORT,
        'image_detail': DIRECT_IMAGE_DETAIL,
        'demonstration_ids': DIRECT_DEMONSTRATION_IDS,
        'demonstration_rule': DIRECT_DEMONSTRATION_RULE,
        'parsing_status': 'valid',
    }
    for key, expected_value in expected.items():
        if record.get(key) != expected_value:
            raise ValueError(f'Incompatible direct cache metadata for {record.get("instance_id")}: {key}')
    record = dict(record)
    record['response'] = validate_structured_direct_response(
        record['response'], str(record['instance_id'])
    )
    return record

def load_or_infer_structured_direct(frame: pd.DataFrame, split: str) -> pd.DataFrame:
    path = direct_cache_path(split)
    cached = index_by_key(load_jsonl_records(path), 'instance_id', str(path))
    for key in list(cached):
        cached[key] = validate_direct_cache_record(cached[key])
    coverage = cache_coverage(frame, cached.values())
    print(f'Direct {split} cache: {coverage["available"]}/{coverage["expected"]}')
    if coverage['missing']:
        print('Missing direct IDs:', coverage['missing'][:20],
              '...' if len(coverage['missing']) > 20 else '')
    if RUN_NEW_API_CALLS:
        v1_by_group = v1_caption_map(frame)
        rows = frame.head(RUN_LIMIT) if RUN_LIMIT is not None else frame
        with tqdm(total=len(rows), desc=f'Structured direct CEA {split}') as progress:
            for _, row in rows.iterrows():
                instance_id = str(row['id'])
                if instance_id not in cached:
                    record = call_structured_direct(
                        row, v1_by_group[str(row['base_id'])], split
                    )
                    append_jsonl(path, record)
                    cached[instance_id] = record
                    time.sleep(REQUEST_SLEEP_SECONDS)
                progress.update(1)
    ordered = [cached[str(value)] for value in frame['id'] if str(value) in cached]
    return pd.DataFrame(ordered)

                                                                                      
train_direct_records = load_or_infer_structured_direct(train_df, 'train')
dev_direct_records = load_or_infer_structured_direct(dev_df, 'dev')


In [ ]:
DIRECT_NUMERIC_FEATURES = [
    'raw_prediction', 'judge_confidence', 'weighted_mean_anchor_score',
    'min_anchor_score', 'near_zero_anchor_fraction', 'n_anchors',
    'generic_substitution_fraction', 'wrong_culture_fraction',
]
DIRECT_CATEGORICAL_FEATURES = ['caption_version_key'] + (
    ['category'] if USE_CATEGORY_FEATURE else []
)

def direct_records_to_features(records: pd.DataFrame, metadata: pd.DataFrame) -> pd.DataFrame:
    output = []
    if records.empty:
        return pd.DataFrame(columns=['id'] + DIRECT_NUMERIC_FEATURES)
    for record in records.to_dict('records'):
        response = record['response']
        anchors = response['anchors']
        if not anchors:
            raise ValueError(f'Empty anchor list for {record["instance_id"]}')
        scores = np.asarray([
            anchor['presence'] * np.sqrt(
                anchor['identity_match'] * anchor['cultural_correctness']
            ) for anchor in anchors
        ], dtype=float)
        importance = np.asarray([anchor['importance'] for anchor in anchors], dtype=float)
        if not np.isfinite(scores).all() or not np.isfinite(importance).all():
            raise ValueError(f'Non-finite anchor features for {record["instance_id"]}')
        total = importance.sum()
        if total <= 0:
            raise ValueError(f'Non-positive importance total for {record["instance_id"]}')
        weights = importance / total
        output.append({
            'id': str(record['instance_id']),
            'raw_prediction': float(response['raw_cea']),
            'judge_confidence': float(response['confidence']),
            'weighted_mean_anchor_score': float(np.dot(weights, scores)),
            'min_anchor_score': float(scores.min()),
            'near_zero_anchor_fraction': float(np.mean(scores <= NEAR_ZERO_THRESHOLD)),
            'n_anchors': int(len(anchors)),
            'generic_substitution_fraction': float(np.mean([
                anchor['generic_substitution'] for anchor in anchors
            ])),
            'wrong_culture_fraction': float(np.mean([
                anchor['wrong_culture_or_landmark'] for anchor in anchors
            ])),
        })
    result = pd.DataFrame(output)
    metadata_columns = ['id', 'base_id', 'caption_version', 'caption_version_key', 'category']
    result = metadata[metadata_columns].merge(result, on='id', how='inner', validate='one_to_one')
    assert not result[DIRECT_NUMERIC_FEATURES].isna().any().any()
    assert result['raw_prediction'].between(0, 1).all()
    return result

train_direct_features = direct_records_to_features(train_direct_records, train_df)
dev_direct_features = direct_records_to_features(dev_direct_records, dev_df)
print('Direct features:', len(train_direct_features), 'train |', len(dev_direct_features), 'dev')
display(train_direct_features.head())


In [ ]:
def safe_spearman(gold, predicted) -> float:
    gold = pd.Series(gold, dtype=float)
    predicted = pd.Series(predicted, dtype=float)
    if len(gold) < 2 or gold.nunique() < 2 or predicted.nunique() < 2:
        return np.nan
    return float(spearmanr(gold, predicted).correlation)

def assert_predictions(values, label: str):
    values = np.asarray(values, dtype=float)
    assert np.isfinite(values).all(), f'{label}: predictions contain NaN/inf'
    assert ((0 <= values) & (values <= 1)).all(), f'{label}: predictions outside [0, 1]'

def make_compact_pipeline(numeric_features, categorical_features, alpha: float):
    preprocessing = ColumnTransformer([
        ('numeric', Pipeline([
            ('imputer', SimpleImputer(strategy='median')),
            ('scale', StandardScaler()),
        ]), numeric_features),
        ('categorical', Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent')),
            ('onehot', OneHotEncoder(handle_unknown='ignore')),
        ]), categorical_features),
    ])
    return Pipeline([('preprocess', preprocessing), ('ridge', Ridge(alpha=alpha))])

def fold_splits(frame: pd.DataFrame, n_splits: int):
    groups = frame['base_id'].astype(str).to_numpy()
    n_splits = min(n_splits, pd.Series(groups).nunique())
    if n_splits < 2:
        raise ValueError('At least two reference-image groups are required')
    splitter = GroupKFold(n_splits=n_splits)
    for train_index, validation_index in splitter.split(frame, groups=groups):
        train_groups = set(groups[train_index])
        validation_groups = set(groups[validation_index])
        assert train_groups.isdisjoint(validation_groups)
        yield train_index, validation_index

def choose_alpha_grouped(frame, numeric_features, categorical_features, alpha_grid=None):
    feature_columns = numeric_features + categorical_features
    candidates = []
    alpha_grid = RIDGE_ALPHAS if alpha_grid is None else list(alpha_grid)
    for alpha in alpha_grid:
        fold_correlations, fold_maes = [], []
        for train_index, validation_index in fold_splits(frame, INNER_FOLDS):
            training = frame.iloc[train_index]
            validation = frame.iloc[validation_index]
            model = make_compact_pipeline(numeric_features, categorical_features, alpha)
            model.fit(training[feature_columns], training['gold_cea'])
            prediction = np.clip(model.predict(validation[feature_columns]), 0, 1)
            fold_correlations.append(safe_spearman(validation['gold_cea'], prediction))
            fold_maes.append(mean_absolute_error(validation['gold_cea'], prediction))
        mean_spearman = np.nanmean(fold_correlations)
        candidates.append({
            'alpha': alpha,
            'mean_spearman': -np.inf if np.isnan(mean_spearman) else mean_spearman,
            'mean_mae': float(np.mean(fold_maes)),
        })
    ranked = sorted(candidates, key=lambda x: (-x['mean_spearman'], x['mean_mae'], x['alpha']))
    return float(ranked[0]['alpha']), pd.DataFrame(candidates)

def empirical_percentile(values, training_reference):
    values = np.asarray(values, dtype=float)
    reference = np.sort(np.asarray(training_reference, dtype=float))
    if len(reference) == 0:
        raise ValueError('Rank transform needs a non-empty training reference')
    return np.searchsorted(reference, values, side='right') / len(reference)

def summarize_fold_metrics(fold_metrics: pd.DataFrame) -> pd.DataFrame:
    rows = []
    for variant, group in fold_metrics.groupby('variant'):
        values = group['spearman'].dropna().astype(float)
        rows.append({
            'variant': variant,
            'cv_spearman_mean': values.mean() if len(values) else np.nan,
            'cv_spearman_std': values.std(ddof=1) if len(values) > 1 else np.nan,
            'cv_spearman_min': values.min() if len(values) else np.nan,
            'cv_spearman_max': values.max() if len(values) else np.nan,
            'cv_mae_mean': group['mae'].mean(),
            'positive_vs_raw_folds': int(group.get('delta_vs_raw', pd.Series(dtype=float)).gt(0).sum()),
            'folds': len(group),
        })
    return pd.DataFrame(rows)

def choose_simple_variant(summary: pd.DataFrame, fold_metrics: pd.DataFrame) -> str:
    priority = {'raw': 0, 'limited_020': 1, 'ridge': 2, 'rank_average': 3}
    raw_mean = float(summary.loc[summary['variant'].eq('raw'), 'cv_spearman_mean'].iloc[0])
    candidate = summary.sort_values(
        ['cv_spearman_mean', 'cv_mae_mean'], ascending=[False, True]
    ).iloc[0]
    name = str(candidate['variant'])
    if name == 'raw' or candidate['cv_spearman_mean'] - raw_mean < 0.01:
        return 'raw'
    deltas = fold_metrics.loc[fold_metrics['variant'].eq(name), 'delta_vs_raw'].dropna()
    stable = (deltas.gt(0).sum() >= int(np.ceil(len(deltas) / 2))) and (deltas.min() > -0.10)
    if not stable:
        return 'raw'
    near_best = summary[
        summary['cv_spearman_mean'] >= candidate['cv_spearman_mean'] - 0.01
    ].copy()
    near_best['priority'] = near_best['variant'].map(priority)
    return str(near_best.sort_values(['priority', 'cv_mae_mean']).iloc[0]['variant'])

def fit_compact_grouped_system(
    frame, numeric_features, categorical_features, label, alpha_grid=None
):
    required = {'id', 'base_id', 'gold_cea', 'raw_prediction'} | set(numeric_features) | set(categorical_features)
    missing = required - set(frame.columns)
    if missing:
        raise ValueError(f'{label}: missing columns {sorted(missing)}')
    assert not frame['gold_cea'].isna().any()
    feature_columns = numeric_features + categorical_features
    oof = {name: np.full(len(frame), np.nan) for name in ['raw', 'ridge', 'limited_020', 'rank_average']}
    fold_rows = []
    for fold, (train_index, validation_index) in enumerate(fold_splits(frame, OUTER_FOLDS), start=1):
        training = frame.iloc[train_index].copy()
        validation = frame.iloc[validation_index].copy()
        assert set(training['base_id']).isdisjoint(set(validation['base_id']))
        alpha, _ = choose_alpha_grouped(
            training, numeric_features, categorical_features, alpha_grid=alpha_grid
        )
        model = make_compact_pipeline(numeric_features, categorical_features, alpha)
        model.fit(training[feature_columns], training['gold_cea'])
        ridge = np.clip(model.predict(validation[feature_columns]), 0, 1)
        raw = validation['raw_prediction'].to_numpy(float)
        limited = np.clip(raw + np.clip(ridge - raw, -RESIDUAL_LIMIT, RESIDUAL_LIMIT), 0, 1)
        training_ridge = np.clip(model.predict(training[feature_columns]), 0, 1)
        rank_average = 0.5 * (
            empirical_percentile(raw, training['raw_prediction'])
            + empirical_percentile(ridge, training_ridge)
        )
        predictions = {'raw': raw, 'ridge': ridge, 'limited_020': limited,
                       'rank_average': rank_average}
        raw_correlation = safe_spearman(validation['gold_cea'], raw)
        for variant, prediction in predictions.items():
            assert_predictions(prediction, f'{label}/{variant}/fold{fold}')
            oof[variant][validation_index] = prediction
            correlation = safe_spearman(validation['gold_cea'], prediction)
            fold_rows.append({
                'system': label, 'fold': fold, 'variant': variant,
                'spearman': correlation,
                'mae': mean_absolute_error(validation['gold_cea'], prediction),
                'delta_vs_raw': correlation - raw_correlation if not np.isnan(correlation) else np.nan,
                'alpha': alpha, 'n_rows': len(validation_index),
                'n_groups': validation['base_id'].nunique(),
                'validation_groups': ','.join(sorted(validation['base_id'].unique())),
            })
    for name, values in oof.items():
        assert_predictions(values, f'{label}/{name}/OOF')
    folds = pd.DataFrame(fold_rows)
    summary = summarize_fold_metrics(folds)
    selected_variant = choose_simple_variant(summary, folds)
    final_alpha, alpha_table = choose_alpha_grouped(
        frame, numeric_features, categorical_features, alpha_grid=alpha_grid
    )
    final_model = make_compact_pipeline(numeric_features, categorical_features, final_alpha)
    final_model.fit(frame[feature_columns], frame['gold_cea'])
    transformed_feature_count = int(final_model.named_steps['preprocess'].transform(frame[feature_columns].head(1)).shape[1])
    return {
        'label': label, 'frame': frame.copy(), 'numeric': numeric_features,
        'categorical': categorical_features, 'feature_columns': feature_columns,
        'oof': oof, 'fold_metrics': folds, 'summary': summary,
        'selected_variant': selected_variant, 'final_alpha': final_alpha,
        'alpha_table': alpha_table, 'final_model': final_model,
        'feature_count': transformed_feature_count,
    }

def predict_compact_system(result, predictors: pd.DataFrame) -> pd.DataFrame:
    assert 'gold_cea' not in predictors.columns, 'Dev/test labels reached prediction code'
    columns = result['feature_columns']
    ridge = np.clip(result['final_model'].predict(predictors[columns]), 0, 1)
    raw = predictors['raw_prediction'].to_numpy(float)
    limited = np.clip(raw + np.clip(ridge - raw, -RESIDUAL_LIMIT, RESIDUAL_LIMIT), 0, 1)
    training = result['frame']
    training_ridge = np.clip(
        result['final_model'].predict(training[columns]), 0, 1
    )
    rank_average = 0.5 * (
        empirical_percentile(raw, training['raw_prediction'])
        + empirical_percentile(ridge, training_ridge)
    )
    output = predictors[['id', 'base_id', 'caption_version', 'caption_version_key', 'category']].copy()
    output['raw'] = raw
    output['ridge'] = ridge
    output['limited_020'] = limited
    output['rank_average'] = rank_average
    output['selected_prediction'] = output[result['selected_variant']]
    for column in ['raw', 'ridge', 'limited_020', 'rank_average', 'selected_prediction']:
        assert_predictions(output[column], f'{result["label"]}/{column}/final')
    return output


In [ ]:
CULTURAL_INVENTORY_PROMPT = r"""
ROLE

You identify persistent culture-bearing anchors for one CRAI-Bench reference
scene. These anchors define the cultural target that must remain evaluable
across all five caption versions, even when a later caption becomes generic.

OFFICIAL CEA QUESTION

Are expected cultural elements present and correctly depicted?

INPUTS

- The reference image.
- The v1 caption, which is the most culturally specific caption for the scene.

CULTURAL SCOPE

Create an inventory only for elements that carry culturally identifying
information. Examples include a traditional garment or face covering, a
recognizable Qatari or Gulf landmark form, a culturally specific performance,
instrument, craft, architectural form, symbol, or customary object.

Exclude ordinary visual and compositional properties unless they are essential
to cultural identity. Do not inventory:

- black-and-white presentation, lighting, weather, water, sky, or moon;
- age, gender, profile view, pose, count, or camera composition;
- generic people, buildings, towers, instruments, boats, or vehicles;
- microscopic facade details, exact texture, exact decoration, or clutter.


CULTURAL KNOWLEDGE FOR QATAR

Qatari identity is not interchangeable with a merely generic Arab or Gulf
appearance. Inspect culture-bearing identity at the appropriate level when it
is supported by the reference and v1. Useful visual families include:

- named landmarks and architectural forms such as Katara Towers, Al Fanar,
  the Museum of Islamic Art, the National Museum of Qatar, Souq Waqif, and
  distinctive Doha skyline structures;
- garments and dress components such as thobe, ghutra, agal, abaya, shayla,
  and battoulah;
- practices and material culture such as falconry, dhow and pearl-diving
  heritage, camel or horse traditions, majlis settings, sadu weaving,
  textiles, and local-market handicrafts.

These are examples, not a checklist. Never invent, add, or require one merely
because the task concerns Qatar. Use it only when the reference/v1 establishes
it. A generic modern tower, generic robe, generic mask, generic market arch,
or generic desert object must not replace a more discriminative Qatari anchor.

ANCHOR RULES

- Return one to six central persistent cultural anchors.
- Each anchor must have two to five atomic core_cultural_features describing
  the minimum visible features needed for recognizable cultural identity.
- core_cultural_features must be broad enough to tolerate normal visual
  variation but discriminative enough to reject a generic substitute.
- optional_surface_features may record minor appearance details for diagnosis;
  they are never scored.
- Use the reference image and v1 together. Apply cultural knowledge cautiously.
- A proper name may appear in established_name only when that exact name occurs
  in v1. Never guess or invent a proper name from visual resemblance.
- If v1 names the anchor, the name remains available to identify the persistent
  target in later generic captions. This does not require pixel-level matching.

OUTPUT JSON

{
  "base_id": "<BASE_ID>",
  "v1_caption": "<V1_CAPTION>",
  "inventory_elements": [
    {
      "inventory_id": "INV_01",
      "element": "<CULTURE-BEARING ANCHOR>",
      "anchor_type": "garment | landmark | architecture | performance | instrument | craft | symbol | customary_object | other",
      "established_name": null,
      "name_source": "none | v1_caption_explicit",
      "core_cultural_features": ["<MINIMUM RECOGNIZABLE CULTURAL FEATURE>"],
      "optional_surface_features": ["<MINOR NON-SCORING DETAIL>"],
      "anchor_evidence": "<WHY THIS ELEMENT CARRIES THE REFERENCE SCENE'S CULTURAL IDENTITY>"
    }
  ],
  "inventory_notes": []
}

Return valid JSON only. Do not inspect the generated image, predict scores,
include generic scene criteria, expose chain-of-thought, use markdown, or
return text outside JSON.
"""


CAPTION_REQUIREMENTS_PROMPT = r"""
ROLE

You construct Cultural Element Accuracy (CEA) requirements for one CRAI-Bench
instance from a shared cultural inventory and the current caption.

OFFICIAL CEA QUESTION

Are expected cultural elements present and correctly depicted?

CULTURAL SCOPE

CEA evaluates culture-bearing elements, not general caption compliance.

The shared reference/v1 inventory defines persistent cultural anchors. Copy
every persistent anchor into the scored requirements for every caption version,
including generic captions, because CEA measures whether cultural identity
survives caption generalization.

The current caption may:

- add a genuinely cultural requirement not already represented by an anchor;
- explicitly request an exact cultural identity;
- determine how much additional detail is reasonably required.

The current caption must not:

- remove a persistent cultural anchor;
- turn ordinary scene properties into CEA criteria;
- allow a generic substitute to receive full cultural credit.

Include only:

1. every persistent cultural anchor from the inventory; and
2. additional caption-explicit cultural requirements.

Exclude black-and-white style, age, gender, profile, pose, count, composition,
water, sky, moon, lighting, generic people, generic buildings, generic vehicles,
and generic instruments. "Instruments are present" is not cultural unless a
cultural instrument identity or culturally specific performance is required.


QATAR-SPECIFICITY CHECK

When the inventory establishes a Qatari landmark, garment, practice, craft, or
customary object, preserve the minimum discriminative identity in the copied
requirement. Do not reduce it to a generic tower, generic Gulf clothing,
generic market, generic animal, or generic instrument. Examples include named
Doha/Qatar landmarks; thobe, ghutra, agal, abaya, shayla, or battoulah;
falconry, dhow and pearl-diving heritage, majlis, sadu/textiles, and local
market handicrafts. These examples are not automatic requirements and must be
used only when supported by the inventory or caption.

PERSISTENT ANCHOR COPYING

For every inventory element create exactly one requirement with:

- criterion_scope = persistent_cultural_anchor;
- inventory_id, element, required_name, core_cultural_features, and
  optional_surface_features copied unchanged from the inventory;
- required_specificity = exact_named when established_name is present,
  otherwise regional;
- importance = 2.0.

An exact-named persistent anchor remains the intended cultural target even when
the current caption omits its name. Exact-named does not imply microscopic or
pixel-level matching; Stage 2 evaluates recognizable identity through the
minimum core cultural features.

CAPTION-EXPLICIT CULTURAL REQUIREMENTS

Add a requirement only when the current caption explicitly introduces a
culture-bearing element not already covered by a persistent anchor. Use:

- criterion_scope = caption_explicit_cultural_requirement;
- inventory_id = null;
- importance = 1.0;
- required_specificity = regional or exact_named;
- required_name only when the exact proper name appears in the current caption.

Keep each core cultural feature atomic. Do not add duplicate or generic
requirements merely to summarize the caption.

OUTPUT JSON

{
  "instance_id": "<INSTANCE_ID>",
  "caption_specificity": "generic | regional | exact_named",
  "requirements": [
    {
      "id": "CEA_01",
      "inventory_id": "INV_01 | null",
      "criterion_scope": "persistent_cultural_anchor | caption_explicit_cultural_requirement",
      "element": "<CULTURE-BEARING ELEMENT>",
      "required_name": null,
      "required_specificity": "regional | exact_named",
      "core_cultural_features": ["<ATOMIC MINIMUM CULTURAL FEATURE>"],
      "optional_surface_features": ["<NON-SCORING MINOR DETAIL>"],
      "importance": 2.0
    }
  ],
  "generation_notes": []
}

Return valid JSON only. Do not inspect the generated image, predict scores,
expose chain-of-thought, use markdown, or return text outside JSON.
"""


STAGE_2_SYSTEM_PROMPT = r"""
ROLE

You are a strict multimodal evaluator of Cultural Element Accuracy (CEA) for
an AI-generated image.

OFFICIAL CEA QUESTION

Are expected cultural elements present and correctly depicted?

TARGET INTERPRETATION

CEA asks whether the culture-bearing elements of the reference scene remain
present and correctly recognizable in the generated image. Consider what the
caption requests, but never allow caption genericization to erase a supplied
persistent cultural anchor.

A generic element may satisfy the literal caption while failing the cultural
anchor. Do not award full CEA merely because a generic caption is satisfied.
For example, generic curved towers do not automatically preserve a culturally
recognizable landmark; generic musicians do not automatically preserve a Gulf
performance; and a generic face covering does not automatically preserve a
traditional battoulah.

At the same time, do not require an exact pixel-level copy, microscopic facade
details, exact texture, exact decoration, exact camera geometry, or minor
optional surface features. Evaluate whether the culture-bearing identity is
recognizable through its minimum core cultural features.


QATAR-SPECIFICITY CHECK

Do not treat "looks Arab/Gulf" as proof of Qatari cultural correctness. When a
supplied requirement establishes a Qatari landmark, architectural identity,
garment, practice, craft, or customary object, compare its discriminative
visible features. Relevant families can include Katara Towers, Al Fanar, the
Museum of Islamic Art, the National Museum of Qatar, Souq Waqif, Doha skyline
structures; thobe, ghutra, agal, abaya, shayla, battoulah; falconry, dhow and
pearl-diving heritage, majlis, sadu/textiles, and local-market handicrafts.
These are examples only. Never require or hallucinate an item not established
by the supplied requirement, reference, and v1 inventory.

INPUT AUTHORITY

- The supplied requirements define the only scored CEA content.
- persistent_cultural_anchor requirements remain scored for every caption.
- caption_explicit_cultural_requirement items add genuinely cultural content.
- The current caption clarifies requested context and specificity but cannot
  remove a persistent anchor.
- The reference image visually clarifies the cultural target; it is not an
  exact-copy target.
- optional_surface_features are diagnostic only and must never lower scores.
- Generic scene compliance is irrelevant unless needed to recognize a supplied
  cultural requirement.

FEATURE SCORING SCALE

Use only 0.00, 0.25, 0.50, 0.75, or 1.00:

- 1.00: culture-bearing element and its defining identity are clearly correct.
- 0.75: recognizable and culturally correct with minor nonessential differences.
- 0.50: partly recognizable but weak, incomplete, or too generic culturally.
- 0.25: faint resemblance with major cultural-feature errors or wrong subtype.
- 0.00: absent, replaced by a generic/noncultural substitute, contradicted, or
  culturally wrong.

Score every core_cultural_feature independently. Do not collapse a multi-feature
requirement into an all-or-nothing identity decision. A missing optional feature
is never a core failure. Do not give presence or correctness credit solely for
ordinary caption fidelity.

COMPONENTS

For every requirement return:

- element_present: whether the culture-bearing element itself is visible;
- one score and concise visible evidence for every core cultural feature;
- cultural_correctness: whether the depicted form remains culturally truthful
  and recognizable at the supplied specificity;
- optional_surface_similarity: diagnostic similarity to optional details, or
  null when no optional details exist;
- evidence: a concise overall explanation grounded in the image pair.

OUTPUT JSON

{
  "instance_id": "<INSTANCE_ID>",
  "requirement_evaluations": [
    {
      "id": "CEA_01",
      "inventory_id": "INV_01 | null",
      "criterion_scope": "<UNCHANGED>",
      "element": "<UNCHANGED>",
      "required_name": null,
      "required_specificity": "<UNCHANGED>",
      "core_cultural_features": ["<UNCHANGED>"],
      "optional_surface_features": ["<UNCHANGED>"],
      "importance": 2.0,
      "element_present": 1.0,
      "core_feature_evaluations": [
        {
          "feature": "<EXACT CORE CULTURAL FEATURE>",
          "score": 1.0,
          "evidence": "<VISIBLE EVIDENCE>"
        }
      ],
      "cultural_correctness": 1.0,
      "optional_surface_similarity": null,
      "evidence": "<OVERALL VISIBLE CULTURAL EVIDENCE>"
    }
  ],
  "evaluation_notes": {
    "missing_inputs": [],
    "unresolved_ambiguities": []
  }
}

Do not return a holistic or final CEA score. Python computes feature coverage,
element scores, and weighted aggregation. Return valid JSON only without
markdown, chain-of-thought, or text outside JSON.
"""


In [ ]:
def image_to_data_url(path: str) -> str:
    path = Path(path)
    mime_by_suffix = {
        '.png': 'image/png', '.jpg': 'image/jpeg',
        '.jpeg': 'image/jpeg', '.webp': 'image/webp',
    }
    mime = mime_by_suffix.get(path.suffix.lower())
    if mime is None:
        raise ValueError(f'Unsupported image type: {path}')
    encoded = base64.b64encode(path.read_bytes()).decode('utf-8')
    return f'data:{mime};base64,{encoded}'

def extract_caption(row: pd.Series) -> str:
    for column in ['caption', 'text', 'prompt']:
        if column in row.index and pd.notna(row[column]):
            return str(row[column])
    ignored = set(ALL_GOLD_COLS + [
        'id', 'split', 'base_id', 'caption_version',
        'ref_image_path', 'generated_image_path',
    ])
    return '\n'.join(
        f'{column}: {row[column]}' for column in row.index
        if column not in ignored and pd.notna(row[column])
    )

def parse_json_object(text: str) -> dict:
    text = text.strip()
    if text.startswith('```'):
        text = re.sub(r'^```(?:json)?\s*', '', text)
        text = re.sub(r'\s*```$', '', text)
    start, end = text.find('{'), text.rfind('}')
    if start >= 0 and end > start:
        text = text[start:end + 1]
    return json.loads(text)

def load_jsonl_records(path: Path) -> list:
    if not path.exists():
        return []
    records = []
    with path.open('r', encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, start=1):
            if not line.strip():
                continue
            try:
                records.append(json.loads(line))
            except json.JSONDecodeError as exc:
                raise ValueError(f'Invalid JSON at {path}:{line_number}') from exc
    return records

def index_by_key(records: list, key: str, source: str) -> dict:
    indexed = {}
    for record in records:
        value = str(record.get(key, ''))
        if not value:
            raise ValueError(f'Missing {key} in {source}')
        if value in indexed:
            raise ValueError(f'Duplicate {key}={value} in {source}')
        indexed[value] = record
    return indexed

def prompt_hash(prompt: str) -> str:
    return hashlib.sha256(prompt.encode('utf-8')).hexdigest()[:10]

def inventory_cache_tag() -> str:
    return f'cea-persistent-inventory_{STAGE_1_MODEL}_prompt-{prompt_hash(CULTURAL_INVENTORY_PROMPT)}'

def inventory_cache_path(split: str) -> Path:
    return CACHE_DIR / f'inventory_{split}_{inventory_cache_tag()}.jsonl'

def normalized_name_tokens(text: str) -> list[str]:
    connectors = {'a', 'an', 'and', 'of', 'or', 'the', 'with'}
    tokens = re.findall(r"[a-z0-9]+", str(text).lower())
    normalized = []
    for token in tokens:
        if token in connectors:
            continue
                                                                              
                                                              
        if len(token) > 3 and token.endswith('s') and not token.endswith('ss'):
            token = token[:-1]
        normalized.append(token)
    return normalized

def caption_supports_name_tokens(name: str, caption: str) -> bool:
    name_tokens = normalized_name_tokens(name)
    caption_tokens = set(normalized_name_tokens(caption))
    return bool(name_tokens) and all(token in caption_tokens for token in name_tokens)

def validate_inventory(inventory: dict, base_id: str, v1_caption: str) -> dict:
    inventory['base_id'] = str(inventory.get('base_id') or base_id)
    if inventory['base_id'] != str(base_id):
        raise ValueError('Inventory returned the wrong base_id')
    elements = inventory.get('inventory_elements')
    if not isinstance(elements, list) or not 1 <= len(elements) <= 6:
        raise ValueError('Inventory must contain 1-6 persistent cultural anchors')

    allowed_types = {
        'garment', 'landmark', 'architecture', 'performance', 'instrument',
        'craft', 'symbol', 'customary_object', 'other',
    }
    seen = set()
    for element in elements:
        required = {
            'inventory_id', 'element', 'anchor_type', 'established_name',
            'name_source', 'core_cultural_features',
            'optional_surface_features', 'anchor_evidence',
        }
        if not required.issubset(element):
            raise ValueError(f'Missing inventory fields: {element}')
        inventory_id = str(element['inventory_id'])
        if inventory_id in seen:
            raise ValueError(f'Duplicate inventory ID: {inventory_id}')
        seen.add(inventory_id)
        if element['anchor_type'] not in allowed_types:
            raise ValueError(f'Invalid cultural anchor type: {element}')
        name = element['established_name']
        source = element['name_source']
        if source not in {'none', 'v1_caption_explicit'}:
            raise ValueError(f'Invalid name_source: {element}')
        if source == 'none' and name is not None:
            raise ValueError('Unestablished inventory names must be null')
        if source == 'v1_caption_explicit':
            if not name:
                raise ValueError('v1_caption_explicit requires an established name')
            exact_match = str(name).lower() in v1_caption.lower()
            if not exact_match:
                if caption_supports_name_tokens(name, v1_caption):
                                                                              
                                                                                 
                                                                            
                    element['established_name'] = None
                    element['name_source'] = 'none'
                else:
                    raise ValueError(
                        'Established name contains tokens unsupported by v1: '
                        f'{name!r}'
                    )
        features = element['core_cultural_features']
        if not isinstance(features, list) or not 2 <= len(features) <= 5:
            raise ValueError('Every anchor needs 2-5 core cultural features')
        if not isinstance(element['optional_surface_features'], list):
            raise ValueError('optional_surface_features must be a list')
        if not str(element['element']).strip() or not str(element['anchor_evidence']).strip():
            raise ValueError('Every anchor needs an element and anchor evidence')
    inventory['v1_caption'] = v1_caption
    return inventory

def generate_inventory(v1_row: pd.Series) -> dict:
    base_id = str(v1_row['base_id'])
    v1_caption = extract_caption(v1_row)
    user_text = f"""Base ID: {base_id}
V1 caption:
{v1_caption}

Build the persistent cultural-anchor inventory from this caption and the
reference image. Exclude generic scene properties. Return JSON only."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = get_openai_client().responses.create(
                model=STAGE_1_MODEL,
                input=[
                    {'role': 'system', 'content': [
                        {'type': 'input_text', 'text': CULTURAL_INVENTORY_PROMPT.strip()}
                    ]},
                    {'role': 'user', 'content': [
                        {'type': 'input_text', 'text': user_text},
                        {'type': 'input_text', 'text': 'Reference image:'},
                        {'type': 'input_image', 'image_url': image_to_data_url(v1_row['ref_image_path'])},
                    ]},
                ],
            )
            return validate_inventory(
                parse_json_object(response.output_text), base_id, v1_caption
            )
        except Exception:
            if attempt == MAX_RETRIES:
                raise
            time.sleep(2 ** attempt)

def generate_inventories_for_split(
    frame: pd.DataFrame, split: str, limit: Optional[int] = None
) -> pd.DataFrame:
    target_rows = frame.head(limit) if limit is not None else frame
    target_base_ids = target_rows['base_id'].astype(str).drop_duplicates().tolist()
    path = inventory_cache_path(split)
    cached = load_jsonl_records(path)
    by_id = index_by_key(cached, 'base_id', str(path))

    with path.open('a', encoding='utf-8') as handle:
        for base_id in tqdm(target_base_ids, desc=f'CEA persistent inventory {split}'):
            v1_rows = frame[
                (frame['base_id'].astype(str) == base_id)
                & (frame['caption_version'] == 1)
            ]
            if len(v1_rows) != 1:
                raise ValueError(f'Expected exactly one v1 caption for {base_id}')
            v1_row = v1_rows.iloc[0]
            if base_id in by_id:
                validate_inventory(by_id[base_id], base_id, extract_caption(v1_row))
                continue
            result = generate_inventory(v1_row)
            handle.write(json.dumps(result, ensure_ascii=False) + '\n')
            handle.flush()
            by_id[base_id] = result
            time.sleep(REQUEST_SLEEP_SECONDS)

    ordered = [by_id[value] for value in target_base_ids]
    print('Inventory configuration:', inventory_cache_tag())
    print('Inventory cache:', path)
    return pd.DataFrame(ordered)


In [ ]:
def requirements_cache_tag() -> str:
    return (
        f'cea-cultural-requirements_{STAGE_1_MODEL}_inventory-{prompt_hash(CULTURAL_INVENTORY_PROMPT)}_'
        f'prompt-{prompt_hash(CAPTION_REQUIREMENTS_PROMPT)}'
    )

def requirements_cache_path(split: str) -> Path:
    return CACHE_DIR / f'requirements_{split}_{requirements_cache_tag()}.jsonl'

def validate_requirements(
    requirements: dict, row: pd.Series, inventory: Optional[dict] = None
) -> dict:
    instance_id = str(row['id'])
    caption = extract_caption(row)
    requirements['instance_id'] = str(requirements.get('instance_id') or instance_id)
    if requirements['instance_id'] != instance_id:
        raise ValueError('Requirements returned the wrong instance_id')
    if requirements.get('caption_specificity') not in {'generic', 'regional', 'exact_named'}:
        raise ValueError('Invalid caption_specificity')

    items = requirements.get('requirements')
    if not isinstance(items, list) or not 1 <= len(items) <= 10:
        raise ValueError('Stage 1 must return 1-10 cultural requirements')

    inventory_by_id = {}
    if inventory is not None:
        inventory_by_id = {
            str(value['inventory_id']): value
            for value in inventory['inventory_elements']
        }

    seen_ids = set()
    persistent_inventory_ids = []
    for item in items:
        required_fields = {
            'id', 'inventory_id', 'criterion_scope', 'element',
            'required_name', 'required_specificity', 'core_cultural_features',
            'optional_surface_features', 'importance',
        }
        if not required_fields.issubset(item):
            raise ValueError(f'Missing requirement fields: {item}')
        item_id = str(item['id'])
        if item_id in seen_ids:
            raise ValueError(f'Duplicate requirement id: {item_id}')
        seen_ids.add(item_id)

        scope = item['criterion_scope']
        if scope not in {
            'persistent_cultural_anchor',
            'caption_explicit_cultural_requirement',
        }:
            raise ValueError(f'Invalid criterion_scope: {item}')
        specificity = item['required_specificity']
        if specificity not in {'regional', 'exact_named'}:
            raise ValueError(f'Invalid required_specificity: {item}')
        features = item['core_cultural_features']
        if not isinstance(features, list) or not 1 <= len(features) <= 5:
            raise ValueError('Each requirement needs 1-5 core cultural features')
        if not isinstance(item['optional_surface_features'], list):
            raise ValueError('optional_surface_features must be a list')

        if scope == 'persistent_cultural_anchor':
            inventory_id = str(item['inventory_id'])
            if inventory_id not in inventory_by_id:
                raise ValueError(f'Unknown persistent inventory_id: {inventory_id}')
            source = inventory_by_id[inventory_id]
            expected = {
                'element': source['element'],
                'required_name': source['established_name'],
                'required_specificity': (
                    'exact_named' if source['established_name'] else 'regional'
                ),
                'core_cultural_features': source['core_cultural_features'],
                'optional_surface_features': source['optional_surface_features'],
            }
            for field, expected_value in expected.items():
                if item[field] != expected_value:
                    raise ValueError(
                        f'Persistent anchor must copy {field} unchanged for {inventory_id}'
                    )
            if float(item['importance']) != 2.0:
                raise ValueError('Persistent cultural anchors must have importance 2.0')
            persistent_inventory_ids.append(inventory_id)
        else:
            if item['inventory_id'] is not None:
                raise ValueError('Caption-explicit additions must have inventory_id=null')
            if float(item['importance']) != 1.0:
                raise ValueError('Caption-explicit cultural requirements must have importance 1.0')
            name = item['required_name']
            if specificity == 'exact_named':
                if not name or str(name).lower() not in caption.lower():
                    raise ValueError(
                        f'Caption-explicit exact name must occur in the caption: {name!r}'
                    )
            elif name is not None:
                raise ValueError('required_name must be null unless exact_named')

        item['importance'] = float(item['importance'])

    if inventory is not None:
        expected_ids = list(inventory_by_id)
        if persistent_inventory_ids != expected_ids:
            raise ValueError(
                'Stage 1 must copy every persistent anchor exactly once and in inventory order; '
                f'expected {expected_ids}, got {persistent_inventory_ids}'
            )
    return requirements

def generate_requirements(row: pd.Series, inventory: dict) -> dict:
    caption = extract_caption(row)
    user_text = f"""Instance ID: {row['id']}
Caption version: v{row['caption_version']}
Current caption:
{caption}

Shared persistent cultural-anchor inventory:
{json.dumps(inventory, ensure_ascii=False, indent=2)}

Copy every persistent anchor and add only caption-explicit cultural
requirements not already covered. Return JSON only."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = get_openai_client().responses.create(
                model=STAGE_1_MODEL,
                input=[
                    {'role': 'system', 'content': [
                        {'type': 'input_text', 'text': CAPTION_REQUIREMENTS_PROMPT.strip()}
                    ]},
                    {'role': 'user', 'content': [
                        {'type': 'input_text', 'text': user_text}
                    ]},
                ],
            )
            return validate_requirements(
                parse_json_object(response.output_text), row, inventory
            )
        except Exception:
            if attempt == MAX_RETRIES:
                raise
            time.sleep(2 ** attempt)

def generate_requirements_for_split(
    frame: pd.DataFrame, split: str, limit: Optional[int] = None
) -> pd.DataFrame:
    rows = frame.head(limit).copy() if limit is not None else frame.copy()
    inventories = index_by_key(
        load_jsonl_records(inventory_cache_path(split)),
        'base_id', str(inventory_cache_path(split)),
    )
    missing_inventory = sorted(set(rows['base_id'].astype(str)) - set(inventories))
    if missing_inventory:
        raise ValueError(f'Missing persistent inventories: {missing_inventory}')

    path = requirements_cache_path(split)
    cached = load_jsonl_records(path)
    by_id = index_by_key(cached, 'instance_id', str(path))
    with path.open('a', encoding='utf-8') as handle:
        for _, row in tqdm(rows.iterrows(), total=len(rows), desc=f'CEA cultural requirements {split}'):
            instance_id = str(row['id'])
            inventory = inventories[str(row['base_id'])]
            if instance_id in by_id:
                validate_requirements(by_id[instance_id], row, inventory)
                continue
            result = generate_requirements(row, inventory)
            handle.write(json.dumps(result, ensure_ascii=False) + '\n')
            handle.flush()
            by_id[instance_id] = result
            time.sleep(REQUEST_SLEEP_SECONDS)

    ordered = [by_id[str(value)] for value in rows['id']]
    print('Requirements configuration:', requirements_cache_tag())
    print('Requirements cache:', path)
    return pd.DataFrame(ordered)


In [ ]:
ALLOWED_COMPONENT_SCORES = {0.0, 0.25, 0.5, 0.75, 1.0}

def stage_2_cache_tag() -> str:
    return (
        f'cea-persistent-coverage_{STAGE_2_MODEL}_reasoning-{STAGE_2_REASONING_EFFORT}_'
        f'detail-{STAGE_2_IMAGE_DETAIL}_inventory-{prompt_hash(CULTURAL_INVENTORY_PROMPT)}_'
        f'requirements-{prompt_hash(CAPTION_REQUIREMENTS_PROMPT)}_'
        f'prompt-{prompt_hash(STAGE_2_SYSTEM_PROMPT)}'
    )

def stage_2_cache_path(split: str) -> Path:
    return CACHE_DIR / f'stage2_{split}_{stage_2_cache_tag()}.jsonl'

def checked_component_score(value, name: str) -> float:
    score = float(value)
    if score not in ALLOWED_COMPONENT_SCORES:
        raise ValueError(f'{name} must use the five-point scale; got {score}')
    return score

def validate_stage_2_evaluation(evaluated: dict, requirements: dict, row: pd.Series) -> dict:
    instance_id = str(row['id'])
    if str(evaluated.get('instance_id')) != instance_id:
        raise ValueError('Stage 2 returned the wrong instance_id')

    originals = requirements['requirements']
    scored_items = evaluated.get('requirement_evaluations')
    if not isinstance(scored_items, list) or len(scored_items) != len(originals):
        raise ValueError('Stage 2 changed the requirement count')

    canonical = {
        'instance_id': instance_id,
        'caption_specificity': requirements['caption_specificity'],
        'requirement_evaluations': [],
    }
    unchanged_fields = [
        'id', 'inventory_id', 'criterion_scope', 'element', 'required_name',
        'required_specificity', 'core_cultural_features',
        'optional_surface_features', 'importance',
    ]
    for position, (original, scored) in enumerate(zip(originals, scored_items), start=1):
        for field in unchanged_fields:
            if scored.get(field) != original.get(field):
                raise ValueError(f'Stage 2 changed requirement {position}: {field}')

        element_present = checked_component_score(
            scored.get('element_present'), f'{original["id"]}.element_present'
        )
        cultural_correctness = checked_component_score(
            scored.get('cultural_correctness'),
            f'{original["id"]}.cultural_correctness',
        )
        feature_evaluations = scored.get('core_feature_evaluations')
        if not isinstance(feature_evaluations, list):
            raise ValueError('core_feature_evaluations must be a list')
        expected_features = original['core_cultural_features']
        returned_features = [value.get('feature') for value in feature_evaluations]
        if returned_features != expected_features:
            raise ValueError(
                f'Stage 2 must evaluate cultural features in unchanged order for {original["id"]}'
            )
        canonical_features = []
        feature_scores = []
        for feature_result in feature_evaluations:
            score = checked_component_score(
                feature_result.get('score'),
                f'{original["id"]}.{feature_result.get("feature")}',
            )
            feature_scores.append(score)
            canonical_features.append({
                'feature': feature_result['feature'],
                'score': score,
                'evidence': str(feature_result.get('evidence', '')).strip(),
            })
        core_coverage = float(np.mean(feature_scores))

                                                                             
                                                                            
                                                                             
                                                                            
        optional_value = scored.get('optional_surface_similarity')
        optional_similarity = None
        optional_surface_evidence = ''
        if isinstance(optional_value, dict):
            optional_surface_evidence = str(
                optional_value.get('evidence', '')
            ).strip()
            optional_value = optional_value.get('score')
        if optional_value is not None:
            try:
                optional_similarity = checked_component_score(
                    optional_value,
                    f'{original["id"]}.optional_surface_similarity',
                )
            except (TypeError, ValueError):
                optional_surface_evidence = str(optional_value).strip()

                                                               
        element_score = float(
            0.25 * element_present
            + 0.50 * core_coverage
            + 0.25 * cultural_correctness
        )
        item = json.loads(json.dumps(original, ensure_ascii=False))
        item.update({
            'element_present': element_present,
            'core_feature_evaluations': canonical_features,
            'core_feature_coverage': core_coverage,
            'cultural_correctness': cultural_correctness,
            'optional_surface_similarity': optional_similarity,
            'optional_surface_evidence': optional_surface_evidence,
            'cea_element_score': element_score,
            'evidence': str(scored.get('evidence', '')).strip(),
        })
        canonical['requirement_evaluations'].append(item)

    notes = evaluated.get('evaluation_notes', {})
    missing = notes.get('missing_inputs', []) if isinstance(notes, dict) else []
    unresolved = notes.get('unresolved_ambiguities', []) if isinstance(notes, dict) else []
    if missing:
        raise ValueError(f'Stage 2 reported missing inputs: {missing}')
    canonical['evaluation_notes'] = {
        'missing_inputs': [],
        'unresolved_ambiguities': unresolved if isinstance(unresolved, list) else [],
    }
    return canonical

def evaluate_requirements(row: pd.Series, inventory: dict, requirements: dict) -> dict:
    user_text = f"""Instance ID: {row['id']}
Current caption:
{extract_caption(row)}

Cultural requirements:
{json.dumps(requirements, ensure_ascii=False, indent=2)}

Shared persistent cultural-anchor inventory:
{json.dumps(inventory, ensure_ascii=False, indent=2)}

The first image is the reference and the second is the generated image.
Evaluate only the supplied cultural requirements. Return JSON only."""
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            response = get_openai_client().responses.create(
                model=STAGE_2_MODEL,
                reasoning={'effort': STAGE_2_REASONING_EFFORT},
                input=[
                    {'role': 'system', 'content': [
                        {'type': 'input_text', 'text': STAGE_2_SYSTEM_PROMPT.strip()}
                    ]},
                    {'role': 'user', 'content': [
                        {'type': 'input_text', 'text': user_text},
                        {'type': 'input_text', 'text': 'REFERENCE IMAGE:'},
                        {'type': 'input_image', 'image_url': image_to_data_url(row['ref_image_path']),
                         'detail': STAGE_2_IMAGE_DETAIL},
                        {'type': 'input_text', 'text': 'AI-GENERATED IMAGE:'},
                        {'type': 'input_image', 'image_url': image_to_data_url(row['generated_image_path']),
                         'detail': STAGE_2_IMAGE_DETAIL},
                    ]},
                ],
            )
            return validate_stage_2_evaluation(
                parse_json_object(response.output_text), requirements, row
            )
        except Exception:
            if attempt == MAX_RETRIES:
                raise
            time.sleep(2 ** attempt)

def evaluate_stage_2_for_split(
    frame: pd.DataFrame, split: str, limit: Optional[int] = None
) -> pd.DataFrame:
    rows = frame.head(limit).copy() if limit is not None else frame.copy()
    inventories = index_by_key(
        load_jsonl_records(inventory_cache_path(split)),
        'base_id', str(inventory_cache_path(split)),
    )
    requirements_by_id = index_by_key(
        load_jsonl_records(requirements_cache_path(split)),
        'instance_id', str(requirements_cache_path(split)),
    )
    requested_ids = rows['id'].astype(str).tolist()
    missing = [value for value in requested_ids if value not in requirements_by_id]
    if missing:
        raise ValueError(f'Missing Stage 1 requirements: {missing[:10]}')

    path = stage_2_cache_path(split)
    cached = index_by_key(load_jsonl_records(path), 'instance_id', str(path))
    with path.open('a', encoding='utf-8') as handle:
        for _, row in tqdm(rows.iterrows(), total=len(rows), desc=f'Stage 2 persistent CEA {split}'):
            instance_id = str(row['id'])
            requirements = requirements_by_id[instance_id]
            if instance_id in cached:
                cached[instance_id] = validate_stage_2_evaluation(
                    cached[instance_id], requirements, row
                )
                continue
            inventory = inventories[str(row['base_id'])]
            result = evaluate_requirements(row, inventory, requirements)
            handle.write(json.dumps(result, ensure_ascii=False) + '\n')
            handle.flush()
            cached[instance_id] = result
            time.sleep(REQUEST_SLEEP_SECONDS)

    ordered = [cached[value] for value in requested_ids]
    print('Stage 2 configuration:', stage_2_cache_tag())
    print('Stage 2 cache:', path)
    return pd.DataFrame(ordered)


In [ ]:
PERSISTENT_NUMERIC_FEATURES = [
    'raw_prediction', 'mean_element_score', 'min_element_score',
    'near_zero_anchor_fraction', 'n_anchors',
]
PERSISTENT_CATEGORICAL_FEATURES = ['caption_version_key'] + (
    ['category'] if USE_CATEGORY_FEATURE else []
)

def load_or_resume_persistent(frame: pd.DataFrame, split: str) -> pd.DataFrame:
    if RUN_NEW_API_CALLS:
        generate_inventories_for_split(frame, split, limit=RUN_LIMIT)
        generate_requirements_for_split(frame, split, limit=RUN_LIMIT)
        return evaluate_stage_2_for_split(frame, split, limit=RUN_LIMIT)
    path = stage_2_cache_path(split)
    records = load_jsonl_records(path)
    coverage = cache_coverage(frame, records)
    print(f'Persistent {split} cache: {coverage["available"]}/{coverage["expected"]}')
    if coverage['missing']:
        print('Missing persistent IDs:', coverage['missing'])
        print('Set RUN_NEW_API_CALLS=True to resume only these missing records.')
    by_id = index_by_key(records, 'instance_id', str(path))
    return pd.DataFrame([by_id[str(value)] for value in frame['id'] if str(value) in by_id])

def persistent_records_to_features(records: pd.DataFrame, metadata: pd.DataFrame) -> pd.DataFrame:
    output = []
    for record in records.to_dict('records'):
        items = record['requirement_evaluations']
        if not items:
            raise ValueError(f'Empty persistent requirement list for {record["instance_id"]}')
        scores = np.asarray([item['cea_element_score'] for item in items], dtype=float)
        weights = np.asarray([item['importance'] for item in items], dtype=float)
        if not np.isfinite(scores).all() or weights.sum() <= 0:
            raise ValueError(f'Malformed persistent features for {record["instance_id"]}')
        output.append({
            'id': str(record['instance_id']),
            'raw_prediction': float(np.average(scores, weights=weights)),
            'mean_element_score': float(scores.mean()),
            'min_element_score': float(scores.min()),
            'near_zero_anchor_fraction': float(np.mean(scores <= NEAR_ZERO_THRESHOLD)),
            'n_anchors': int(len(scores)),
        })
    raw = pd.DataFrame(output)
    metadata_columns = ['id', 'base_id', 'caption_version', 'caption_version_key', 'category']
    result = metadata[metadata_columns].merge(raw, on='id', how='inner', validate='one_to_one')
    if len(result):
        assert not result[PERSISTENT_NUMERIC_FEATURES].isna().any().any()
        assert_predictions(result['raw_prediction'], 'persistent/raw')
    return result

train_persistent_records = load_or_resume_persistent(train_df, 'train')
dev_persistent_records = load_or_resume_persistent(dev_df, 'dev')
train_persistent_features = persistent_records_to_features(train_persistent_records, train_df)
dev_persistent_features = persistent_records_to_features(dev_persistent_records, dev_df)
print('Persistent features:', len(train_persistent_features), 'train |', len(dev_persistent_features), 'dev')


In [ ]:
ENSEMBLE_WEIGHTS = [0.25, 0.50, 0.75]                                       

def fold_metrics_for_prediction(frame, prediction, label):
    rows = []
    for fold, (_, validation_index) in enumerate(fold_splits(frame, OUTER_FOLDS), start=1):
        gold = frame.iloc[validation_index]['gold_cea']
        pred = np.asarray(prediction)[validation_index]
        rows.append({'variant': label, 'fold': fold,
                     'spearman': safe_spearman(gold, pred),
                     'mae': mean_absolute_error(gold, pred)})
    return pd.DataFrame(rows)

def fit_optional_ensemble(direct_result, persistent_result):
    direct = direct_result['frame'][['id', 'base_id', 'gold_cea']].copy()
    direct['direct_oof'] = direct_result['oof'][direct_result['selected_variant']]
    persistent = persistent_result['frame'][['id']].copy()
    persistent['persistent_oof'] = persistent_result['oof'][persistent_result['selected_variant']]
    aligned = direct.merge(persistent, on='id', how='inner', validate='one_to_one')
    if len(aligned) != len(direct) or len(aligned) != len(persistent):
        raise ValueError('Ensemble requires identical complete training rows')
    rows, predictions = [], {}
    for weight in ENSEMBLE_WEIGHTS:
        name = f'weighted_{weight:.2f}'
        prediction = weight * aligned['direct_oof'] + (1 - weight) * aligned['persistent_oof']
        predictions[name] = prediction.to_numpy()
        metrics = fold_metrics_for_prediction(aligned, prediction, name)
        metrics['weight_direct'] = weight
        rows.append(metrics)
    equal_raw = 0.5 * (
        direct_result['oof']['raw'] + persistent_result['oof']['raw']
    )
    predictions['equal_raw_average'] = equal_raw
    metrics = fold_metrics_for_prediction(aligned, equal_raw, 'equal_raw_average')
    metrics['weight_direct'] = 0.5
    rows.append(metrics)
    rank_average = 0.5 * (
        rankdata(direct_result['oof'][direct_result['selected_variant']], method='average') / len(aligned)
        + rankdata(persistent_result['oof'][persistent_result['selected_variant']], method='average') / len(aligned)
    )
    predictions['equal_rank_average'] = rank_average
    metrics = fold_metrics_for_prediction(aligned, rank_average, 'equal_rank_average')
    metrics['weight_direct'] = 0.5
    rows.append(metrics)
    fold_table = pd.concat(rows, ignore_index=True)
    summary = fold_table.groupby('variant').agg(
        cv_spearman_mean=('spearman', 'mean'),
        cv_spearman_std=('spearman', 'std'),
        cv_spearman_min=('spearman', 'min'),
        cv_spearman_max=('spearman', 'max'),
        cv_mae_mean=('mae', 'mean'),
    ).reset_index()
    direct_variant = direct_result['selected_variant']
    persistent_variant = persistent_result['selected_variant']
    direct_folds = direct_result['fold_metrics'].query(
        'variant == @direct_variant'
    ).sort_values('fold')['spearman'].to_numpy()
    persistent_folds = persistent_result['fold_metrics'].query(
        'variant == @persistent_variant'
    ).sort_values('fold')['spearman'].to_numpy()
    stronger = np.maximum(direct_folds, persistent_folds)
    best = summary.sort_values(['cv_spearman_mean', 'cv_mae_mean'], ascending=[False, True]).iloc[0]
    best_name = str(best['variant'])
    candidate_folds = fold_table.query('variant == @best_name').sort_values('fold')['spearman'].to_numpy()
    deltas = candidate_folds - stronger
    justified = bool((deltas.mean() > 0.01) and
                     (np.sum(deltas > 0) >= int(np.ceil(len(deltas) / 2))) and
                     (deltas.min() > -0.10))
    return {'aligned': aligned, 'fold_metrics': fold_table, 'summary': summary,
            'predictions': predictions, 'selected_variant': best_name if justified else None,
            'justified': justified}


In [ ]:
def add_gold(features: pd.DataFrame, gold_frame: pd.DataFrame) -> pd.DataFrame:
    result = features.merge(
        gold_frame[['id', 'CRAI_CEA']], on='id', how='inner', validate='one_to_one'
    ).rename(columns={'CRAI_CEA': 'gold_cea'})
    return result

def predictors_only(features: pd.DataFrame) -> pd.DataFrame:
    result = features.copy()
    assert 'CRAI_CEA' not in result.columns and 'gold_cea' not in result.columns
    return result

def evaluate_final(gold_frame, prediction_frame, prediction_column, label):
    merged = gold_frame[['id', 'base_id', 'caption_version', 'CRAI_CEA']].merge(
        prediction_frame[['id', prediction_column]], on='id', how='inner',
        validate='one_to_one'
    )
    prediction = merged[prediction_column].astype(float)
    assert_predictions(prediction, label)
    return {
        'system': label,
        'dev_spearman': safe_spearman(merged['CRAI_CEA'], prediction),
        'dev_mae': mean_absolute_error(merged['CRAI_CEA'], prediction),
        'dev_rows': len(merged), 'dev_groups': merged['base_id'].nunique(),
    }, merged.rename(columns={'CRAI_CEA': 'gold', prediction_column: 'predicted'})

def by_caption_version(evaluation_frame, system):
    rows = []
    for version, group in evaluation_frame.groupby('caption_version'):
        rows.append({'system': system, 'caption_version': f'v{int(version)}',
                     'spearman': safe_spearman(group['gold'], group['predicted']),
                     'mae': mean_absolute_error(group['gold'], group['predicted']),
                     'n': len(group)})
    return pd.DataFrame(rows)

def cluster_bootstrap_spearman(evaluation_frame, replicates=BOOTSTRAP_REPLICATES):
    groups = evaluation_frame['base_id'].astype(str).unique()
    if len(groups) < 4:
        return (np.nan, np.nan)
    rng = np.random.default_rng(RANDOM_SEED)
    values = []
    indexed = {group: evaluation_frame[evaluation_frame['base_id'].astype(str).eq(group)]
               for group in groups}
    for _ in range(replicates):
        sampled = rng.choice(groups, size=len(groups), replace=True)
        draw = pd.concat([indexed[group] for group in sampled], ignore_index=True)
        value = safe_spearman(draw['gold'], draw['predicted'])
        if not np.isnan(value):
            values.append(value)
    return tuple(np.quantile(values, [0.025, 0.975])) if values else (np.nan, np.nan)

def caption_median_baseline(train_metadata, dev_metadata):
    training = train_metadata[['id', 'base_id', 'caption_version', 'CRAI_CEA']].copy()
    training = training.rename(columns={'CRAI_CEA': 'gold_cea'})
    oof = np.full(len(training), np.nan)
    fold_rows = []
    for fold, (train_index, validation_index) in enumerate(fold_splits(training, OUTER_FOLDS), start=1):
        medians = training.iloc[train_index].groupby('caption_version')['gold_cea'].median()
        prediction = training.iloc[validation_index]['caption_version'].map(medians).to_numpy(float)
        oof[validation_index] = prediction
        gold = training.iloc[validation_index]['gold_cea']
        fold_rows.append({'fold': fold, 'spearman': safe_spearman(gold, prediction),
                          'mae': mean_absolute_error(gold, prediction)})
    medians = training.groupby('caption_version')['gold_cea'].median()
    dev_prediction = dev_metadata[['id']].copy()
    dev_prediction['prediction'] = dev_metadata['caption_version'].map(medians).to_numpy(float)
    folds = pd.DataFrame(fold_rows)
    return {'oof': oof, 'folds': folds, 'dev_prediction': dev_prediction}

def historical_direct_features(path: Path) -> pd.DataFrame:
    output = []
    for record in load_jsonl_records(path):
        scores = np.asarray([x['preservation_score'] for x in record['cultural_elements']], float)
        output.append({'id': str(record['instance_id']),
                       'direct_raw_cea': float(record['raw_cea']),
                       'direct_confidence': float(record['confidence']),
                       'direct_mean_element': scores.mean(),
                       'direct_min_element': scores.min(),
                       'direct_zero_fraction': np.mean(np.isclose(scores, 0)),
                       'direct_n_elements': len(scores)})
    return pd.DataFrame(output)

def historical_persistent_features(records: pd.DataFrame) -> pd.DataFrame:
    output = []
    for record in records.to_dict('records'):
        items = record['requirement_evaluations']
        scores = np.asarray([x['cea_element_score'] for x in items], float)
        weights = np.asarray([x['importance'] for x in items], float)
        output.append({'id': str(record['instance_id']),
                       'persistent_cea_decomposed': np.average(scores, weights=weights),
                       'persistent_mean_element_present': np.mean([x['element_present'] for x in items]),
                       'persistent_mean_core_coverage': np.mean([x['core_feature_coverage'] for x in items]),
                       'persistent_mean_cultural_correctness': np.mean([x['cultural_correctness'] for x in items]),
                       'persistent_pct_zero_elements': np.mean(np.isclose(scores, 0))})
    return pd.DataFrame(output)

HISTORICAL_DIRECT_TRAIN = HISTORICAL_DIRECT_CACHE_DIR / (
    'train_direct-cultural-cea_gpt-5.5_reasoning-medium_detail-original_'
    'prompt-68f12070e9_demos-a43c6b0d44.jsonl'
)
HISTORICAL_DIRECT_DEV = HISTORICAL_DIRECT_CACHE_DIR / (
    'dev_direct-cultural-cea_gpt-5.5_reasoning-medium_detail-original_'
    'prompt-68f12070e9_demos-a43c6b0d44.jsonl'
)

def reproduce_historical_hybrid_d():
    if not HISTORICAL_DIRECT_TRAIN.exists() or not HISTORICAL_DIRECT_DEV.exists():
        print('Historical direct cache missing; using saved source-notebook benchmark only.')
        return None
    direct_train = historical_direct_features(HISTORICAL_DIRECT_TRAIN)
    direct_dev = historical_direct_features(HISTORICAL_DIRECT_DEV)
    persistent_train = historical_persistent_features(train_persistent_records)
    persistent_dev = historical_persistent_features(dev_persistent_records)
    eligible = train_df[~train_df['base_id'].isin({'img_022', 'img_035'})].copy()
    def table(metadata, direct, persistent, include_gold):
        columns = ['id', 'base_id', 'caption_version', 'caption_version_key', 'category']
        if include_gold:
            columns.append('CRAI_CEA')
        result = metadata[columns].merge(direct, on='id', validate='one_to_one').merge(
            persistent, on='id', validate='one_to_one'
        )
        for version in range(1, 6):
            mask = result['caption_version'].eq(version).astype(float)
            result[f'direct_x_v{version}'] = result['direct_raw_cea'] * mask
            result[f'persistent_x_v{version}'] = result['persistent_cea_decomposed'] * mask
        result['raw_prediction'] = result['direct_raw_cea']
        if include_gold:
            result = result.rename(columns={'CRAI_CEA': 'gold_cea'})
        return result
    train_table = table(eligible, direct_train, persistent_train, True)
    dev_table = table(dev_df, direct_dev, persistent_dev, False)
    numeric = [
        'direct_raw_cea', 'persistent_cea_decomposed',
        'persistent_mean_element_present', 'persistent_mean_core_coverage',
        'persistent_mean_cultural_correctness', 'persistent_pct_zero_elements',
        'caption_version',
    ] + [f'direct_x_v{x}' for x in range(1, 6)] + [f'persistent_x_v{x}' for x in range(1, 6)]
    categorical = ['category']
    result = fit_compact_grouped_system(
        train_table, numeric, categorical, 'Historical Hybrid D',
        alpha_grid=[0.1, 0.3, 1.0, 3.0, 10.0, 30.0, 100.0],
    )
    prediction = predict_compact_system(result, dev_table)
                                                                                      
    prediction['selected_prediction'] = prediction['ridge']
    result['selected_variant'] = 'ridge'
    return result, prediction

comparison_rows = []
version_reports = []
evaluation_frames = {}
system_results = {}
fold_report_frames = []

                                  
median_result = caption_median_baseline(train_df, dev_df)
metric, evaluation = evaluate_final(dev_df, median_result['dev_prediction'], 'prediction', 'Caption-version median')
folds = median_result['folds']
fold_report_frames.append(folds.assign(system='Caption-version median', variant='median'))
comparison_rows.append({
    **metric, 'feature_count': 1, 'calibration_method': 'train caption-version median',
    'cv_spearman_mean': folds['spearman'].mean(), 'cv_spearman_std': folds['spearman'].std(ddof=1),
    'cv_spearman_min': folds['spearman'].min(), 'cv_spearman_max': folds['spearman'].max(),
    'train_groups': train_df['base_id'].nunique(), 'train_rows': len(train_df),
    'notes': 'Grouped baseline; train labels only',
})
version_reports.append(by_caption_version(evaluation, 'Caption-version median'))
evaluation_frames['Caption-version median'] = evaluation

                                                 
historical = reproduce_historical_hybrid_d()
if historical is not None:
    historical_result, historical_prediction = historical
    metric, evaluation = evaluate_final(dev_df, historical_prediction, 'selected_prediction', 'Historical Hybrid D')
    summary = historical_result['summary'].query("variant == 'ridge'").iloc[0]
    comparison_rows.append({
        **metric, 'feature_count': historical_result['feature_count'],
        'calibration_method': f'Previous interaction Ridge (alpha={historical_result["final_alpha"]:g})',
        'cv_spearman_mean': summary['cv_spearman_mean'], 'cv_spearman_std': summary['cv_spearman_std'],
        'cv_spearman_min': summary['cv_spearman_min'], 'cv_spearman_max': summary['cv_spearman_max'],
        'train_groups': historical_result['frame']['base_id'].nunique(),
        'train_rows': len(historical_result['frame']),
        'notes': 'Historical: incompatible gold-demo cache; img_022/img_035 excluded',
    })
    version_reports.append(by_caption_version(evaluation, 'Historical Hybrid D'))
    evaluation_frames['Historical Hybrid D'] = evaluation
    system_results['historical'] = historical_result
    fold_report_frames.append(historical_result['fold_metrics'].copy())
else:
    comparison_rows.append({
        'system': 'Historical Hybrid D', 'feature_count': 17,
        'calibration_method': 'Previous interaction Ridge',
        'cv_spearman_mean': np.nan, 'cv_spearman_std': np.nan,
        'cv_spearman_min': np.nan, 'cv_spearman_max': np.nan,
        'dev_spearman': 0.7437, 'dev_mae': 0.2159,
        'train_groups': 22, 'train_rows': 110, 'dev_groups': 8, 'dev_rows': 40,
        'notes': 'Saved source-notebook result; cache unavailable to reproduce folds',
    })

def run_compact_system(name, train_features, dev_features, numeric, categorical, complete_required=True):
    train_complete = len(train_features) == len(train_df)
    dev_complete = len(dev_features) == len(dev_df)
    if not dev_complete or (complete_required and not train_complete):
        print(f'{name} skipped: train {len(train_features)}/{len(train_df)}, dev {len(dev_features)}/{len(dev_df)}')
        expected_feature_count = len(numeric) + 5 + (train_df['category'].nunique() if 'category' in categorical else 0)
        reason = (
            f'PENDING: compatible cache coverage train {len(train_features)}/{len(train_df)}, '
            f'dev {len(dev_features)}/{len(dev_df)}'
        )
        comparison_rows.extend([
            {
                'system': f'Raw {name}', 'feature_count': 1,
                'calibration_method': 'none', 'notes': reason,
                'train_groups': train_features['base_id'].nunique() if 'base_id' in train_features else 0,
                'train_rows': len(train_features), 'dev_groups': dev_features['base_id'].nunique() if 'base_id' in dev_features else 0,
                'dev_rows': len(dev_features),
            },
            {
                'system': f'Compact calibrated {name}', 'feature_count': expected_feature_count,
                'calibration_method': 'compact grouped Ridge', 'notes': reason,
                'train_groups': train_features['base_id'].nunique() if 'base_id' in train_features else 0,
                'train_rows': len(train_features), 'dev_groups': dev_features['base_id'].nunique() if 'base_id' in dev_features else 0,
                'dev_rows': len(dev_features),
            },
        ])
        return None
    if not train_complete:
        print(f'WARNING: {name} is provisional because its train cache is incomplete.')
    training = add_gold(train_features, train_df)
    result = fit_compact_grouped_system(training, numeric, categorical, name)
    fold_report_frames.append(result['fold_metrics'].copy())
    prediction = predict_compact_system(result, predictors_only(dev_features))
    for variant, display_name in [('raw', f'Raw {name}'),
                                  (result['selected_variant'], f'Compact calibrated {name}')]:
        if display_name in [row['system'] for row in comparison_rows]:
            continue
        metric, evaluation = evaluate_final(dev_df, prediction, variant, display_name)
        summary = result['summary'].query('variant == @variant').iloc[0]
        method = {
            'raw': 'none',
            'ridge': f'Ridge (alpha={result["final_alpha"]:g})',
            'limited_020': f'Ridge with ±0.20 residual limit (alpha={result["final_alpha"]:g})',
            'rank_average': f'Raw/Ridge rank average (alpha={result["final_alpha"]:g})',
        }[variant]
        comparison_rows.append({
            **metric, 'feature_count': 1 if variant == 'raw' else result['feature_count'],
            'calibration_method': method,
            'cv_spearman_mean': summary['cv_spearman_mean'], 'cv_spearman_std': summary['cv_spearman_std'],
            'cv_spearman_min': summary['cv_spearman_min'], 'cv_spearman_max': summary['cv_spearman_max'],
            'train_groups': training['base_id'].nunique(), 'train_rows': len(training),
            'notes': ('Complete grouped training' if train_complete else 'PROVISIONAL: 10 cached train rows missing'),
        })
        version_reports.append(by_caption_version(evaluation, display_name))
        evaluation_frames[display_name] = evaluation
    result['dev_prediction'] = prediction
    result['train_complete'] = train_complete
    return result

direct_result = run_compact_system(
    'direct', train_direct_features, dev_direct_features,
    DIRECT_NUMERIC_FEATURES, DIRECT_CATEGORICAL_FEATURES, complete_required=True,
)
persistent_result = run_compact_system(
    'persistent', train_persistent_features, dev_persistent_features,
    PERSISTENT_NUMERIC_FEATURES, PERSISTENT_CATEGORICAL_FEATURES,
    complete_required=not ALLOW_INCOMPLETE_CACHE_DIAGNOSTICS,
)
system_results['direct'] = direct_result
system_results['persistent'] = persistent_result

comparison_table = pd.DataFrame(comparison_rows)
for index, row in comparison_table.iterrows():
    frame = evaluation_frames.get(row['system'])
    if frame is not None:
        low, high = cluster_bootstrap_spearman(frame)
        comparison_table.loc[index, 'dev_spearman_cluster_ci95'] = f'[{low:.3f}, {high:.3f}]'

column_order = [
    'system', 'feature_count', 'calibration_method',
    'cv_spearman_mean', 'cv_spearman_std', 'cv_spearman_min', 'cv_spearman_max',
    'dev_spearman', 'dev_mae', 'dev_spearman_cluster_ci95',
    'train_groups', 'train_rows', 'dev_groups', 'dev_rows', 'notes',
]
comparison_table = comparison_table.reindex(columns=column_order)
display(comparison_table.round(4))
caption_version_table = pd.concat(version_reports, ignore_index=True) if version_reports else pd.DataFrame()
all_fold_metrics = pd.concat(fold_report_frames, ignore_index=True, sort=False)
display(caption_version_table.round(4))
display(all_fold_metrics.round(4))

comparison_table.to_csv(OUTPUT_DIR / 'cea_system_comparison.tsv', sep='\t', index=False)
caption_version_table.to_csv(OUTPUT_DIR / 'cea_by_caption_version.tsv', sep='\t', index=False)
all_fold_metrics.to_csv(OUTPUT_DIR / 'cea_grouped_cv_fold_metrics.tsv', sep='\t', index=False)


In [ ]:
                                                                                                     
ensemble_result = None
if (direct_result is not None and persistent_result is not None and
        direct_result['train_complete'] and persistent_result['train_complete']):
    ensemble_result = fit_optional_ensemble(direct_result, persistent_result)
    display(ensemble_result['summary'].round(4))
    ensemble_result['fold_metrics'].to_csv(
        OUTPUT_DIR / 'cea_ensemble_grouped_cv_folds.tsv', sep='\t', index=False
    )
    if ensemble_result['justified']:
        variant = ensemble_result['selected_variant']
        direct_dev = direct_result['dev_prediction']['selected_prediction'].to_numpy()
        persistent_dev = persistent_result['dev_prediction']['selected_prediction'].to_numpy()
        if variant.startswith('weighted_'):
            weight = float(variant.split('_')[1])
            values = weight * direct_dev + (1 - weight) * persistent_dev
        elif variant == 'equal_raw_average':
            values = 0.5 * (direct_result['dev_prediction']['raw'] + persistent_result['dev_prediction']['raw'])
        else:
            values = 0.5 * (
                empirical_percentile(direct_dev, direct_result['oof'][direct_result['selected_variant']])
                + empirical_percentile(persistent_dev, persistent_result['oof'][persistent_result['selected_variant']])
            )
        ensemble_dev_prediction = dev_df[['id']].copy()
        ensemble_dev_prediction['prediction'] = values
        metric, evaluation = evaluate_final(dev_df, ensemble_dev_prediction, 'prediction', 'Optional ensemble')
        best = ensemble_result['summary'].query('variant == @variant').iloc[0]
        ensemble_row = {
            **metric, 'feature_count': direct_result['feature_count'] + persistent_result['feature_count'],
            'calibration_method': variant,
            'cv_spearman_mean': best['cv_spearman_mean'], 'cv_spearman_std': best['cv_spearman_std'],
            'cv_spearman_min': best['cv_spearman_min'], 'cv_spearman_max': best['cv_spearman_max'],
            'train_groups': train_df['base_id'].nunique(), 'train_rows': len(train_df),
            'notes': 'Used: stable grouped-CV improvement',
        }
        low, high = cluster_bootstrap_spearman(evaluation)
        ensemble_row['dev_spearman_cluster_ci95'] = f'[{low:.3f}, {high:.3f}]'
        comparison_table = pd.concat([comparison_table, pd.DataFrame([ensemble_row])], ignore_index=True)
        evaluation_frames['Optional ensemble'] = evaluation
        ensemble_version = by_caption_version(evaluation, 'Optional ensemble')
        caption_version_table = pd.concat(
            [caption_version_table, ensemble_version], ignore_index=True
        )
        caption_version_table.to_csv(
            OUTPUT_DIR / 'cea_by_caption_version.tsv', sep='\t', index=False
        )
    else:
        print('Ensemble rejected: grouped training CV did not show stable improvement.')
        best = ensemble_result['summary'].sort_values(
            ['cv_spearman_mean', 'cv_mae_mean'], ascending=[False, True]
        ).iloc[0]
        comparison_table = pd.concat([comparison_table, pd.DataFrame([{
            'system': 'Optional ensemble (rejected)',
            'feature_count': direct_result['feature_count'] + persistent_result['feature_count'],
            'calibration_method': best['variant'],
            'cv_spearman_mean': best['cv_spearman_mean'],
            'cv_spearman_std': best['cv_spearman_std'],
            'cv_spearman_min': best['cv_spearman_min'],
            'cv_spearman_max': best['cv_spearman_max'],
            'train_groups': train_df['base_id'].nunique(), 'train_rows': len(train_df),
            'notes': 'Rejected before dev: grouped-CV stability rule failed',
        }])], ignore_index=True)
else:
    print('Ensemble pending: complete compatible direct and persistent train caches are required.')
    comparison_table = pd.concat([comparison_table, pd.DataFrame([{
        'system': 'Optional ensemble',
        'feature_count': np.nan,
        'calibration_method': 'pending grouped-CV selection',
        'train_groups': min(
            len(train_direct_features['base_id'].unique()) if 'base_id' in train_direct_features else 0,
            len(train_persistent_features['base_id'].unique()) if 'base_id' in train_persistent_features else 0,
        ),
        'train_rows': min(len(train_direct_features), len(train_persistent_features)),
        'notes': 'PENDING: complete compatible direct and persistent caches required',
    }])], ignore_index=True)

display(comparison_table.round(4))
comparison_table.to_csv(OUTPUT_DIR / 'cea_system_comparison.tsv', sep='\t', index=False)


In [ ]:
def normalized_rank_error(gold, predicted):
    n = len(gold)
    if n < 2:
        return np.zeros(n)
    gold_rank = rankdata(gold, method='average') / n
    predicted_rank = rankdata(predicted, method='average') / n
    return np.abs(predicted_rank - gold_rank)

diagnostics = dev_df[
    ['id', 'base_id', 'caption_version', 'category', 'CRAI_CEA',
     'ref_image_path', 'generated_image_path']
].rename(columns={'CRAI_CEA': 'gold_cea'}).copy()

def merge_prediction_column(frame, result, source, name):
    if result is None:
        frame[name] = np.nan
        return frame
    values = result['dev_prediction'][['id', source]].rename(columns={source: name})
    return frame.merge(values, on='id', how='left', validate='one_to_one')

diagnostics = merge_prediction_column(diagnostics, direct_result, 'raw', 'raw_direct_prediction')
diagnostics = merge_prediction_column(diagnostics, direct_result, 'selected_prediction', 'calibrated_direct_prediction')
diagnostics = merge_prediction_column(diagnostics, persistent_result, 'raw', 'raw_persistent_prediction')
diagnostics = merge_prediction_column(diagnostics, persistent_result, 'selected_prediction', 'calibrated_persistent_prediction')

if ensemble_result is not None and ensemble_result.get('justified') and 'Optional ensemble' in evaluation_frames:
    ensemble_values = evaluation_frames['Optional ensemble'][['id', 'predicted']].rename(
        columns={'predicted': 'ensemble_prediction'}
    )
    diagnostics = diagnostics.merge(ensemble_values, on='id', how='left', validate='one_to_one')
else:
    diagnostics['ensemble_prediction'] = np.nan

if direct_result is not None:
    diagnostics['calibration_residual'] = (
        diagnostics['calibrated_direct_prediction'] - diagnostics['raw_direct_prediction']
    )
    primary_column = 'calibrated_direct_prediction'
elif persistent_result is not None:
    diagnostics['calibration_residual'] = (
        diagnostics['calibrated_persistent_prediction'] - diagnostics['raw_persistent_prediction']
    )
    primary_column = 'calibrated_persistent_prediction'
elif 'Historical Hybrid D' in evaluation_frames:
    values = evaluation_frames['Historical Hybrid D'][['id', 'predicted']]
    diagnostics = diagnostics.merge(values, on='id', how='left', validate='one_to_one')
    diagnostics['calibration_residual'] = np.nan
    primary_column = 'predicted'
else:
    primary_column = None

if primary_column:
    diagnostics['absolute_error'] = (diagnostics[primary_column] - diagnostics['gold_cea']).abs()
    diagnostics['rank_error'] = normalized_rank_error(diagnostics['gold_cea'], diagnostics[primary_column])
else:
    diagnostics['absolute_error'] = np.nan
    diagnostics['rank_error'] = np.nan

diagnostic_columns = [
    'id', 'base_id', 'caption_version', 'gold_cea',
    'raw_direct_prediction', 'calibrated_direct_prediction',
    'raw_persistent_prediction', 'calibrated_persistent_prediction',
    'ensemble_prediction', 'calibration_residual', 'absolute_error', 'rank_error',
]
display(diagnostics[diagnostic_columns].sort_values('absolute_error', ascending=False).head(20).round(4))

                                                                               
                                                                                   
large_corrections = []
for label, result in [('direct', direct_result), ('persistent', persistent_result)]:
    if result is not None:
        prediction = result['dev_prediction']
        changed = prediction.assign(
            correction=prediction['ridge'] - prediction['raw']
        ).loc[lambda x: x['correction'].abs() > 0.20]
        if len(changed):
            changed = changed.assign(system=label)
            large_corrections.append(changed)
large_correction_table = pd.concat(large_corrections, ignore_index=True) if large_corrections else pd.DataFrame()
display(Markdown('### Corrections larger than 0.20'))
display(large_correction_table.round(4))

display(Markdown('### Targeted `img_034_v2` inspection'))
display(diagnostics.loc[diagnostics['id'].eq('img_034_v2'), diagnostic_columns].round(4))

diagnostics.to_csv(OUTPUT_DIR / 'cea_per_example_diagnostics.tsv', sep='\t', index=False)
large_correction_table.to_csv(OUTPUT_DIR / 'cea_large_calibration_corrections.tsv', sep='\t', index=False)


In [ ]:
def show_diagnostic_cases(frame, count=8):
    selected = frame.sort_values('absolute_error', ascending=False).head(count)
    for _, row in selected.iterrows():
        fig, axes = plt.subplots(1, 2, figsize=(12, 4.8))
        axes[0].imshow(plt.imread(row['ref_image_path']))
        axes[0].set_title('Reference')
        axes[1].imshow(plt.imread(row['generated_image_path']))
        axes[1].set_title('Generated')
        for axis in axes:
            axis.axis('off')
        fig.suptitle(
            f"{row['id']} | gold={row['gold_cea']:.3f} | "
            f"error={row['absolute_error']:.3f}"
        )
        plt.tight_layout()
        plt.show()
        if direct_result is not None:
            record = dev_direct_records.loc[
                dev_direct_records['instance_id'].astype(str).eq(str(row['id'])), 'response'
            ]
            if len(record):
                display(pd.DataFrame(record.iloc[0]['anchors']))
                print(record.iloc[0]['brief_reason'])

show_diagnostic_cases(diagnostics, count=8)


In [ ]:
def conclusion_text():
    available = comparison_table.dropna(subset=['dev_spearman']).copy()
    direct_ready = direct_result is not None
    persistent_ready = persistent_result is not None and persistent_result.get('train_complete', False)
    lines = ['### Evidence-based conclusion', '']
    if direct_ready:
        direct_raw = comparison_table.query("system == 'Raw direct'")
        direct_cal = comparison_table.query("system == 'Compact calibrated direct'")
        historical_row = comparison_table.query("system == 'Historical Hybrid D'")
        lines.append(
            f"- Structured direct versus Hybrid D: direct dev Spearman is "
            f"{direct_cal['dev_spearman'].iloc[0]:.3f}; historical Hybrid D is "
            f"{historical_row['dev_spearman'].iloc[0]:.3f}. This comparison is interpreted "
            f"together with grouped-CV mean/std, not from dev alone."
        )
        improvement = direct_cal['cv_spearman_mean'].iloc[0] - direct_raw['cv_spearman_mean'].iloc[0]
        lines.append(f"- Compact calibration versus raw direct: grouped-CV mean change is {improvement:+.3f}.")
        selected = direct_result['selected_variant']
        ridge_mean = direct_result['summary'].query("variant == 'ridge'")['cv_spearman_mean'].iloc[0]
        limited_mean = direct_result['summary'].query("variant == 'limited_020'")['cv_spearman_mean'].iloc[0]
        lines.append(
            f"- Residual limiting protects extremes mechanically by capping each Ridge correction at ±0.20. "
            f"Its grouped-CV mean Spearman is {limited_mean:.3f} versus {ridge_mean:.3f} for unrestricted "
            f"Ridge; `{selected}` was selected by the predefined stability rule."
        )
    else:
        lines.append('- Structured direct versus Hybrid D: **not yet determined**; the new prompt/schema cache is missing.')
        lines.append('- Compact calibration and residual limiting for direct CEA: **not yet determined**.')
    if persistent_ready:
        lines.append('- Persistent-anchor contribution: evaluated on all training groups and eligible for ensemble testing.')
    elif persistent_result is not None:
        lines.append('- Persistent-anchor contribution: provisional only; ten training judgments from two groups are still missing.')
    else:
        lines.append('- Persistent-anchor contribution: unavailable until compatible caches are complete.')
    if ensemble_result is not None:
        lines.append(
            '- Ensemble: ' + ('justified by the predefined grouped-CV stability rule.'
                              if ensemble_result['justified'] else
                              'not justified; use the stronger single system.')
        )
    else:
        lines.append('- Ensemble: pending complete compatible judgments for both systems.')

    eligible_names = []
    if direct_ready:
        eligible_names.append('Compact calibrated direct')
    if persistent_ready:
        eligible_names.append('Compact calibrated persistent')
    if ensemble_result is not None and ensemble_result.get('justified'):
        eligible_names.append('Optional ensemble')
    if eligible_names:
        candidates = comparison_table[comparison_table['system'].isin(eligible_names)].copy()
        recommendation = candidates.sort_values(
            ['cv_spearman_mean', 'cv_spearman_std'], ascending=[False, True]
        ).iloc[0]
        lines.append(f"- Recommended submission configuration: **{recommendation['system']}**, "
                     f"chosen from training grouped CV. Freeze it before blind-test inference.")
    else:
        lines.append('- Recommended submission configuration: **pending**. Do not select from dev until compatible caches are complete.')
    lines.append('- Uncertainty: dev has only eight independent reference groups; its cluster-bootstrap interval and per-version correlations are correspondingly wide/unstable.')
    lines.append('')
    lines.append('The old Hybrid D caches and saved benchmark remain untouched and are historical only.')
    return '\n'.join(lines)

display(Markdown(conclusion_text()))
display(comparison_table.round(4))
print('Saved comparison:', OUTPUT_DIR / 'cea_system_comparison.tsv')
print('Saved diagnostics:', OUTPUT_DIR / 'cea_per_example_diagnostics.tsv')
